# Phase 4 — Comparaison de modèles et évaluation (FD001, puis FD002–FD004)

> **En 30 secondes**
> - **Question** : quels modèles prédisent le mieux la durée de vie restante (RUL) d'un moteur, quelles améliorations tiennent, et où nous situer face aux articles publiés ?
> - **Méthode** : les mêmes 5 plis de validation croisée par moteur pour tous les modèles ; une amélioration n'est gardée que si elle passe une règle écrite avant les résultats ; le test officiel est évalué une seule fois par modèle final, contre la RUL brute et contre la RUL plafonnée à 125.
> - **Amélioration retenue (F)** : la variable `ecartinit_20` de Matisse (écart de chaque capteur à sa moyenne des 20 premiers cycles) baisse le RMSE de validation FD001 des 5 modèles qui apprennent, de 4,76 cycles (LSTM : 13,70 → 8,94) à 7,51 cycles (XGBoost : 16,87 → 9,36). La fenêtre de 50 cycles pour les réseaux n'est pas retenue.
> - **Test officiel FD001** (RMSE contre la RUL brute / plafonnée) : LSTM 12,29 / 11,37 ; XGBoost 13,10 / 11,88, contre 19,70 / 18,53 pour XGBoost Phase 3.
> - **Protocole (G)** : les 5 modèles qui apprennent battent les 5 baselines (la meilleure, Ridge, a un RMSE de validation de 16,95) ; mais sur les 100 moteurs test, aucune paire de modèles appris n'est significativement différente.
> - **Littérature (H)** : si les articles plafonnent aussi la vérité test (ce que leurs chiffres sur FD002 et FD004 suggèrent fortement), notre LSTM FD001 (11,37) est au niveau de DAST (11,43) et STAGNN (11,50). Mais notre chiffre vient d'un seul entraînement (11,26 ± 0,43 sur 3 graines), le leur d'une moyenne de 10.
> - **Prudence (C)** : pénaliser la surestimation dans l'apprentissage (α de 1 à 3) baisse de 33 % les lignes de validation que XGBoost Phase 3 surestime de plus de 20 cycles, au prix d'un RMSE de validation de 17,68 au lieu de 16,86.
> - **Limite principale** : les choix sont faits sur FD001 et appliqués tels quels ailleurs ; sur FD004, le CNN 1D retenu se dégrade fortement au test (score NASA 446 374, 55 moteurs sur 248 surestimés de plus de 20 cycles).

**Sommaire**

*Partie 1 — modèles à réglages de base (46 variables de la Phase 3), calculés en direct*
- [A. Préparation et règles](#partie-a)
- [B. Comparaison à réglages de base](#partie-b)
- [C. Où mettre la pénalité des surestimations ?](#partie-c)
- [D. Évaluation sur le test officiel](#partie-d)
- [E. Les moteurs surestimés de plus de 20 cycles](#partie-e)

*Partie 2 — versions améliorées, relues dans les résultats des scripts*
- [F. Améliorations testées](#partie-f)
- [G. Évaluation selon le protocole](#partie-g)
- [H. Où nous situer dans la littérature](#partie-h)

*Partie 3*
- [I. Synthèse](#partie-i)
- [J. Démo : tester un moteur que les modèles n'ont jamais vu](#partie-j)

**Exécution** : quelques minutes de bout en bout (durées affichées à la fin des sections I et J). Les sections F à H relisent les fichiers de `sorties/resultats/` ; s'il en manque, la cellule F.1 affiche la commande qui les produit. Les chiffres de l'encadré sont vérifiés par le code en I.5, ceux des conclusions F, G et H par une cellule de contrôle à la fin de chaque section.

<a id="partie-a"></a>
## A. Préparation et règles

### A.1 Quatre règles fixées avant tout résultat

1. **Le test officiel ne sert jamais à choisir** (ni features, ni modèle, ni réglage, ni seuil) : sinon son score deviendrait optimiste. Ici, il sert seulement à mesurer les modèles de la section B (section D), les versions retenues en section F (F.7, G) et à la démonstration de la section C.1.
2. **Les mêmes 5 plis pour tous les modèles** (`GroupKFold` par moteur, graine 42, comme en Phase 3) : un écart entre deux modèles ne peut donc pas venir du découpage.
3. **Le score NASA d'évaluation garde ses constantes officielles 13 / 10** : c'est la note, on n'y touche pas.
4. **La prudence se règle dans l'apprentissage** : pour réduire les surestimations, on change la perte que le modèle minimise (section C), jamais la note.

Un résultat non significatif ou mixte est écrit comme tel.

### A.2 Imports et réglages

**Ce qu'on fait** : on importe les fonctions du dossier `src/` au lieu de recopier le pipeline ici, et on fixe les seuils utilisés partout.
**Les modules** : `features.py` prépare les données (Phases 1 à 3), `evaluation.py` calcule les métriques, la validation croisée et les tests appariés, `config_modele.py` fabrique un modèle à partir d'un petit dictionnaire de réglages.

In [ ]:
import sys
sys.path.insert(0, "..")  # le dossier parent de notebooks/ (racine du depot) devient importable : "from src ..." marche

import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # masque un avertissement d'affichage, sans effet sur les calculs

# bibliotheques standard de Python
import copy  # deepcopy : copie independante d'un dictionnaire (l'original reste intact)
import time  # chronometres

# calcul, tableaux, figures, tests statistiques
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats  # correlation de Spearman, test de Mann-Whitney

# fonctions du projet (dossier src/)
from src.config_modele import CONFIG_PHASE3, construire_fabrique_depuis_config
from src.evaluation import (
    calculer_plis,                          # les 5 plis de validation croisee par moteur
    comparer_apparie,                       # test t apparie + test de Wilcoxon signe
    demontrer_constantes_score_degenerees,  # demonstration de la section C.1
    erreurs_par_moteur,                     # RMSE moteur par moteur
    evaluer_en_validation_croisee,          # entraine et evalue un modele sur les 5 plis
    evaluer_sur_test,                       # evalue un modele deja entraine sur le test officiel
    nb_surestimes,                          # nombre de predictions surestimees de plus d'un seuil
    rmse,                                   # racine de l'erreur quadratique moyenne
    score_nasa,                             # score asymetrique NASA (13 / 10)
)
from src.features import preparer_test, preparer_train

SEUIL_SURESTIMATION = 20  # cycles : une surestimation de plus de 20 cycles est comptee comme dangereuse (choix Phase 3)
SEUIL_ALERTE = 30         # cycles : RUL predite <= 30 -> envoi en maintenance ; HYPOTHESE DE TRAVAIL, pas une valeur officielle
PLAFOND_RUL = 125         # plafond de la RUL (Phase 2) : la cible d'entrainement ne depasse jamais 125

# couleurs des figures : 3 teintes bien distinctes, toujours la meme teinte pour la meme chose d'une figure
# a l'autre ; le texte et les reperes restent en gris fonce ou en noir
COULEUR_1 = "#2a78d6"  # bleu
COULEUR_2 = "#eb6834"  # orange
COULEUR_3 = "#1baf7a"  # vert d'eau
COULEUR_TEXTE = "#52514e"  # gris fonce (reperes, annotations)
plt.rcParams["axes.grid"] = True  # quadrillage sur toutes les figures
plt.rcParams["axes.axisbelow"] = True  # quadrillage dessine sous les barres et les points
plt.rcParams["grid.color"] = "#e4e3df"  # quadrillage gris clair, discret
plt.rcParams["axes.spines.top"] = False  # pas de bord en haut des graphiques
plt.rcParams["axes.spines.right"] = False  # pas de bord a droite

pd.set_option("display.width", 220)  # tableaux affiches sur une ligne plus large
pd.set_option("display.max_columns", 40)  # toutes les colonnes affichees
debut_notebook = time.perf_counter()  # chronometre : depart du calcul des sections A a F

### A.3 Données d'entraînement et de test

**Ce qu'on fait** : `preparer_train` refait le pipeline des Phases 1 à 3 sur `train_FD001.txt` (capteurs constants retirés, mise à l'échelle min-max, lissage, RUL plafonnée à 125, features glissantes) ; `preparer_test` applique au test les **mêmes** réglages, appris sur le train, sans rien réapprendre.
**Comment lire le résultat** : 20 631 lignes (une par cycle) et 46 features côté entraînement ; 100 lignes côté test, une par moteur au dernier cycle observé, à comparer à la RUL vraie de `RUL_FD001.txt`.

In [ ]:
contexte_train = preparer_train(fenetre=5, plafond=PLAFOND_RUL)  # pipeline des Phases 1 a 3 sur le train (statistiques sur 5 cycles)
contexte_test = preparer_test(contexte_train)  # memes transformations sur le test, avec les reglages appris sur le train

X = contexte_train["X"]            # les 46 features, une ligne par cycle
y = contexte_train["y"]            # la cible : RUL plafonnee a 125
groups = contexte_train["groups"]  # numero de moteur de chaque ligne (sert a GroupKFold)

X_test_dernier = contexte_test["X_test_dernier"]  # features du dernier cycle observe de chaque moteur test
y_test_vraie = contexte_test["y_test_vraie"]      # RUL vraie BRUTE de RUL_FD001.txt (non plafonnee)

print(f"Train : {X.shape[0]} lignes, {X.shape[1]} features, {groups.nunique()} moteurs")
print(f"Test  : {X_test_dernier.shape[0]} moteurs (un point par moteur, au dernier cycle observe)")
assert X.shape == (20631, 46)  # garde-fou : memes dimensions qu'en Phase 3
assert X_test_dernier.shape == (100, 46)  # 100 moteurs test, memes 46 features

### A.4 Les 5 plis de validation croisée, calculés une seule fois

**Ce qu'on fait** : on répartit les 100 moteurs d'entraînement en 5 groupes de 20 ; chaque groupe sert une fois de validation pendant que les 4 autres servent à apprendre.
**Pourquoi par moteur** : un moteur entier est soit en apprentissage, soit en validation ; sinon le modèle verrait des cycles voisins du même moteur des deux côtés et paraîtrait meilleur qu'il n'est.
**Comment lire le résultat** : 20 moteurs en validation dans chaque pli ; `calculer_plis` vérifie aussi qu'aucun moteur n'est des deux côtés. Ces plis servent à tous les modèles du notebook.

In [ ]:
plis = calculer_plis(X, y, groups, n_splits=5, random_state=42)  # 5 plis par moteur, melange de graine 42 (comme en Phase 3)

NB_MOTEURS_TRAIN = int(groups.nunique())  # 100 moteurs d'entrainement
for numero_pli, (idx_train, idx_val) in enumerate(plis):  # idx_train, idx_val : positions des lignes de chaque cote
    nb_moteurs_val = groups.iloc[idx_val].nunique()  # nombre de moteurs mis de cote pour la validation dans ce pli
    print(f"Pli {numero_pli + 1} : {len(idx_train)} lignes d'entrainement, {len(idx_val)} lignes de validation "
          f"({nb_moteurs_val} moteurs)")

### A.5 Les colonnes des tableaux

`resumer_cv` transforme le résultat d'une validation croisée en une ligne de tableau, toujours avec les mêmes colonnes :

| Colonne | Sens |
|---|---|
| `rmse_cv_moyen`, `rmse_cv_ecart_type` | erreur (en cycles) moyenne sur les 5 plis, et sa variation d'un pli à l'autre |
| `mae_cv_moyen` | erreur absolue moyenne, moins sensible aux grosses erreurs isolées |
| `nasa_cv_moyen` | moyenne des scores NASA (13/10) des 5 plis ; environ 4 000 lignes par pli, d'où des valeurs proches de 30 000 (le score du test ne porte que sur 100 points) |
| `surestimes_oof` | lignes surestimées de plus de 20 cycles parmi les 20 631 prédictions hors-pli (OOF : chaque ligne est prédite par le modèle du pli où elle était en validation) |
| `alertes_manquees_oof` | RUL vraie ≤ 30 mais prédiction > 30 : le moteur aurait dû partir en maintenance (dangereux) |
| `fausses_alertes_oof` | RUL vraie > 30 mais prédiction ≤ 30 : maintenance inutile (coûteux, pas dangereux) |
| `biais_oof` | moyenne de (prédiction − vérité) ; négatif = le modèle sous-estime en moyenne (prudent) |
| `temps_entrainement_5_plis_s`, `prediction_ms_par_moteur` | entraînement des 5 plis (s) ; prédiction de toute la trajectoire d'un moteur (ms) |

`resumer_test` fait de même pour le test. `metriques_par_moteur` calcule RMSE, pénalité NASA et surestimations **moteur par moteur**, pour les tests appariés.

In [ ]:
def resumer_cv(nom, resultats, nb_moteurs):
    """
    Resume les resultats d'une validation croisee en une ligne de tableau (un dictionnaire).

    Parameters
    ----------
    nom : str
        Le nom du modele (premiere colonne du tableau).
    resultats : dict
        Le dictionnaire renvoye par `evaluer_en_validation_croisee`.
    nb_moteurs : int
        Nombre de moteurs predits au total (100), pour le temps de prediction par moteur.

    Returns
    -------
    dict
        Les colonnes communes a tous les tableaux de validation croisee de ce notebook.
    """
    # on recopie les metriques utiles, une cle a la fois (sens de chaque colonne : section A.5)
    ligne = {}
    ligne["modele"] = nom
    ligne["rmse_cv_moyen"] = resultats["rmse_moyen"]
    ligne["rmse_cv_ecart_type"] = resultats["rmse_ecart_type"]
    ligne["mae_cv_moyen"] = resultats["mae_moyen"]
    ligne["nasa_cv_moyen"] = resultats["nasa_moyen"]
    ligne["surestimes_oof"] = resultats["surestimes_oof"]
    ligne["alertes_manquees_oof"] = resultats["alertes_manquees_oof"]
    ligne["fausses_alertes_oof"] = resultats["fausses_alertes_oof"]
    ligne["biais_oof"] = resultats["biais_oof"]
    ligne["temps_entrainement_5_plis_s"] = resultats["temps_entrainement_total"]
    # temps de prediction total des 5 plis (secondes) divise par le nombre de moteurs, converti en millisecondes
    ligne["prediction_ms_par_moteur"] = 1000.0 * resultats["temps_prediction_total"] / nb_moteurs
    return ligne


def resumer_test(nom, resultats_test):
    """
    Resume l'evaluation d'un modele sur le test officiel en une ligne de tableau (un dictionnaire).

    Parameters
    ----------
    nom : str
        Le nom du modele.
    resultats_test : dict
        Le dictionnaire renvoye par `evaluer_sur_test`.

    Returns
    -------
    dict
        RMSE (contre la RUL brute et contre la RUL plafonnee), MAE, score NASA, surestimes > 20,
        alertes manquees et fausses alertes, sur les 100 moteurs test.
    """
    # memes metriques que resumer_cv, mais sur les 100 moteurs test (un point par moteur)
    ligne = {}
    ligne["modele"] = nom
    ligne["rmse_test"] = resultats_test["rmse"]
    ligne["rmse_test_rul_plafonnee"] = resultats_test["rmse_contre_rul_plafonnee"]
    ligne["mae_test"] = resultats_test["mae"]
    ligne["nasa_test"] = resultats_test["score_nasa"]
    ligne["surestimes_test"] = resultats_test["nb_surestimes"]
    ligne["alertes_manquees_test"] = resultats_test["alertes_manquees"]
    ligne["fausses_alertes_test"] = resultats_test["fausses_alertes"]
    return ligne


def metriques_par_moteur(y_true, y_pred, groups):
    """
    Calcule trois metriques moteur par moteur : RMSE, penalite NASA (13/10) et surestimations > 20.

    Parameters
    ----------
    y_true : array-like
        Les vraies valeurs (une par ligne).
    y_pred : array-like
        Les predictions (une par ligne, meme ordre).
    groups : array-like
        Le numero de moteur de chaque ligne (sur le test : une seule ligne par moteur).

    Returns
    -------
    pandas.DataFrame
        Une ligne par moteur (triee par numero) : "unit_number", "rmse", "nasa" (somme des penalites
        NASA des lignes du moteur) et "surestimes" (lignes surestimees de plus de 20 cycles).
    """
    vrais = np.asarray(y_true, dtype=float)  # tableaux numpy : on pourra les filtrer avec un masque
    predits = np.asarray(y_pred, dtype=float)
    groupes = np.asarray(groups)

    lignes = []
    for moteur_id in np.unique(groupes):  # un moteur a la fois, dans l'ordre croissant des numeros
        masque = groupes == moteur_id  # Vrai sur les lignes de ce moteur
        ligne = {}
        ligne["unit_number"] = moteur_id
        ligne["rmse"] = rmse(vrais[masque], predits[masque])
        ligne["nasa"] = score_nasa(vrais[masque], predits[masque])
        ligne["surestimes"] = nb_surestimes(vrais[masque], predits[masque], seuil=SEUIL_SURESTIMATION)
        lignes.append(ligne)
    return pd.DataFrame(lignes)

<a id="partie-b"></a>
## B. Comparaison à réglages de base, sur les mêmes plis

**Ce qu'on fait** : on évalue quatre modèles avec des réglages par défaut (aucun n'est optimisé, pour comparer à armes égales). **Pourquoi** : vérifier si XGBoost, choisi en Phase 3, fait mieux que des alternatives.

| Modèle | Réglages |
|---|---|
| Référence naïve | prédit toujours la moyenne de la cible d'entraînement : la borne basse |
| Random Forest | 200 arbres, profondeur non bornée (défaut scikit-learn) |
| XGBoost Phase 3 | 200 arbres, profondeur 5, pas 0,05 (exactement la Phase 3) |
| LightGBM | 200 arbres, pas 0,05, 31 feuilles par arbre (défaut LightGBM), mode déterministe |

**Contrôle de reproduction** : XGBoost Phase 3 doit redonner un RMSE CV moyen de 16,87 et un score NASA CV moyen de 29 782,6 (le document de Phase 3 écrit 16,86 / 29 729,3, obtenus dans un autre environnement : voir sa note de reproductibilité).

In [ ]:
# chaque modele est decrit par un petit dictionnaire (format de src/config_modele.py) : famille, perte
# d'apprentissage, parametres de la perte, hyperparametres et nom lisible
CONFIG_REFERENCE_NAIVE = {
    "famille": "reference_naive", "perte": "native", "parametres_perte": {}, "hyperparametres": {},
    "nom": "Reference naive (moyenne)",
}
CONFIG_RANDOM_FOREST = {
    "famille": "random_forest", "perte": "native", "parametres_perte": {},
    "hyperparametres": {"n_estimators": 200, "random_state": 42, "n_jobs": -1},  # 200 arbres, graine 42, tous les coeurs
    "nom": "Random Forest (base)",
}
CONFIG_XGB_PHASE3 = copy.deepcopy(CONFIG_PHASE3)  # exactement la Phase 3 (copie : l'original reste intact)
CONFIG_LIGHTGBM_BASE = {
    "famille": "lightgbm", "perte": "native", "parametres_perte": {},
    "hyperparametres": {"n_jobs": 4},  # 4 coeurs ; le reste : valeurs de fabrique_lightgbm (200 arbres, pas 0.05, ...)
    "nom": "LightGBM (base)",
}

# les 4 modeles compares, dans l'ordre des tableaux
CONFIGS_BASE = {}
CONFIGS_BASE["Reference naive"] = CONFIG_REFERENCE_NAIVE
CONFIGS_BASE["Random Forest"] = CONFIG_RANDOM_FOREST
CONFIGS_BASE["XGBoost Phase 3"] = CONFIG_XGB_PHASE3
CONFIGS_BASE["LightGBM"] = CONFIG_LIGHTGBM_BASE

resultats_cv_base = {}  # nom du modele -> dictionnaire renvoye par evaluer_en_validation_croisee
for nom_modele in CONFIGS_BASE:  # un modele a la fois, toujours sur les memes plis
    fabrique = construire_fabrique_depuis_config(CONFIGS_BASE[nom_modele])  # fonction qui renvoie un modele neuf a chaque appel
    debut = time.perf_counter()  # chronometre de ce modele
    resultats_cv_base[nom_modele] = evaluer_en_validation_croisee(  # 5 entrainements, un par pli
        fabrique, X, y, groups, plis, seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE
    )
    duree = time.perf_counter() - debut
    print(f"{nom_modele:16s} : RMSE CV moyen = {resultats_cv_base[nom_modele]['rmse_moyen']:.2f} ({duree:.1f} s)")

# controle de reproduction de la Phase 3 dans cet environnement : le notebook s'arrete si les chiffres changent
assert round(resultats_cv_base["XGBoost Phase 3"]["rmse_moyen"], 2) == 16.87
assert round(resultats_cv_base["XGBoost Phase 3"]["nasa_moyen"], 1) == 29782.6

### B.1 Tableau comparatif (validation croisée)

**Comment lire** : une ligne par modèle, colonnes expliquées en A.5 ; plus petit = mieux, sauf `biais_oof` (le plus proche de 0 = sans biais). Les colonnes de temps changent d'une exécution à l'autre, les autres non.

In [ ]:
lignes = []  # une ligne de tableau par modele
for nom_modele in resultats_cv_base:
    lignes.append(resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN))
tableau_cv_base = pd.DataFrame(lignes).set_index("modele")  # le nom du modele sert d'etiquette de ligne
tableau_cv_base.round(2)  # affichage arrondi a 2 decimales (les valeurs exactes restent dans tableau_cv_base)

### B.2 Les écarts sont-ils significatifs ? Tests appariés par moteur

**Notion — test apparié** : on compare deux modèles **moteur par moteur** (les mêmes 100 moteurs), pas seulement leurs moyennes. Exemple : sur un moteur difficile, RMSE 25 (Random Forest) et 24 (XGBoost) ; sur un moteur facile, 10 et 11. Les RMSE vont de 10 à 25, mais les différences (+1, −1) restent petites : c'est sur elles que porte le test, la difficulté propre à chaque moteur s'annule.
- **Test t apparié** : la différence moyenne (modèle − XGBoost) est-elle nulle ? **Wilcoxon signé** : même question avec les rangs, sans supposer de loi normale.
- **p-valeur** : probabilité d'observer un écart au moins aussi grand si les deux modèles étaient en réalité équivalents. Petite p-valeur = écart difficile à attribuer au hasard.
- **Bonferroni** : 3 comparaisons, donc on exige p < 0,05 / 3 ≈ 0,017 pour chacune (risque global de fausse découverte gardé à 5 %).

**Comment lire** : différence moyenne négative = le modèle a un RMSE par moteur plus petit que XGBoost.

In [ ]:
# RMSE hors-pli de chaque moteur, pour chaque modele (100 lignes, triees par numero de moteur)
erreurs_moteur_base = {}
for nom_modele in resultats_cv_base:
    erreurs_moteur_base[nom_modele] = erreurs_par_moteur(y, resultats_cv_base[nom_modele]["predictions_oof"], groups)

rmse_moteur_xgb = erreurs_moteur_base["XGBoost Phase 3"]["rmse"].to_numpy()  # reference des comparaisons
SEUIL_BONFERRONI = 0.05 / 3  # 3 comparaisons contre XGBoost Phase 3 : seuil de 0,017 pour chacune

lignes = []
for nom_modele in resultats_cv_base:
    if nom_modele == "XGBoost Phase 3":
        continue  # on ne compare pas XGBoost a lui-meme
    rmse_moteur_modele = erreurs_moteur_base[nom_modele]["rmse"].to_numpy()  # 100 RMSE, memes moteurs, meme ordre
    comparaison = comparer_apparie(rmse_moteur_modele, rmse_moteur_xgb)  # tests sur les 100 differences modele - XGBoost
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["n_moteurs"] = comparaison["n"]  # nombre de paires comparees
    ligne["difference_moyenne_rmse_par_moteur"] = comparaison["difference_moyenne"]  # negatif = modele meilleur
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(rmse_moteur_modele < rmse_moteur_xgb))  # compte les Vrai
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    # significatif seulement si les DEUX tests passent sous le seuil de Bonferroni
    ligne["significatif_bonferroni"] = (comparaison["p_valeur_t"] < SEUIL_BONFERRONI) and (
        comparaison["p_valeur_wilcoxon"] < SEUIL_BONFERRONI)
    lignes.append(ligne)
tableau_tests_base = pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)")
tableau_tests_base.round(4)  # 4 decimales : 0.0000 veut dire p < 0,00005

### B.3 Figure de comparaison

**Comment lire** : un critère par panneau (jamais deux échelles sur le même axe). Panneau RMSE : la barre est la moyenne des 5 plis, les points sont les 5 plis. La référence naïve, hors échelle, est citée dans le titre.

In [ ]:
noms_compares = ["Random Forest", "XGBoost Phase 3", "LightGBM"]  # la reference naive est hors echelle : pas dessinee
positions = np.arange(len(noms_compares))  # positions 0, 1, 2 des barres sur l'axe horizontal

fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # 4 panneaux : 2 lignes x 2 colonnes

# (1) RMSE CV : moyenne (barre) et valeur de chaque pli (points)
ax = axes[0, 0]
for position, nom_modele in zip(positions, noms_compares):  # zip : une position et un nom a chaque tour
    resultats = resultats_cv_base[nom_modele]
    ax.bar(position, resultats["rmse_moyen"], width=0.6, color=COULEUR_1, alpha=0.85)
    ax.scatter(np.full(5, position), resultats["rmse_par_pli"], color="black", s=18, zorder=3)  # 5 points a la meme abscisse
    ax.text(position, 19.5, f"moyenne {resultats['rmse_moyen']:.2f}", ha="center", fontsize=9)  # au-dessus des points
ax.set_ylim(14, 20)
ax.set_ylabel("RMSE (cycles)")
ax.set_title("RMSE CV moyen (barre) et par pli (points)")

# (2) score NASA CV moyen
ax = axes[0, 1]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["nasa_moyen"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_1, alpha=0.85)
    ax.text(position, valeur * 1.01, f"{valeur:,.0f}".replace(",", " "), ha="center", fontsize=9)  # milliers separes par une espace
ax.set_ylabel("score NASA (13/10), moyenne des 5 plis")
ax.set_title("Score NASA CV moyen (plus bas = mieux)")

# (3) surestimations dangereuses : lignes OOF surestimees de plus de 20 cycles
ax = axes[1, 0]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["surestimes_oof"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_2, alpha=0.85)
    ax.text(position, valeur * 1.01, str(valeur), ha="center", fontsize=9)
ax.set_ylabel("lignes OOF (sur 20 631)")
ax.set_title("Surestimés de plus de 20 cycles (OOF)")

# (4) temps d'entrainement cumule sur les 5 plis
ax = axes[1, 1]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["temps_entrainement_total"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_TEXTE, alpha=0.85)
    ax.text(position, valeur * 1.02, f"{valeur:.1f} s", ha="center", fontsize=9)
ax.set_ylabel("secondes (5 plis)")
ax.set_title("Temps d'entraînement (5 plis)")

for ax in axes.ravel():  # ravel : les 4 panneaux a la suite ; memes etiquettes de modeles sous chacun
    ax.set_xticks(positions)
    ax.set_xticklabels(noms_compares)

rmse_naive = resultats_cv_base["Reference naive"]["rmse_moyen"]
fig.suptitle(f"Phase 4 — comparaison à réglages de base, mêmes 5 plis GroupKFold "
             f"(référence naïve hors échelle : RMSE CV = {rmse_naive:.1f})")
plt.tight_layout()  # evite que les titres et etiquettes se chevauchent
plt.show()

### B.4 Conclusion de la section B

- **Les modèles apprennent** : RMSE CV de 41,7 pour la référence naïve contre environ 16,9 pour les trois ensembles d'arbres (p < 0,0001).
- **Précision : égalité statistique** entre Random Forest (16,96), XGBoost Phase 3 (16,87) et LightGBM (16,85) : 0,11 cycle d'écart au plus, contre un écart-type d'environ 1,3 d'un pli à l'autre ; tests appariés non significatifs (p = 0,78 / 0,58 et 0,79 / 0,80).
- **Prudence : pas de différence nette non plus** (surestimés > 20 : 2 302 / 2 384 / 2 316 ; alertes manquées : 565 / 552 / 568).
- **Temps** : XGBoost et LightGBM s'entraînent en quelques secondes pour les 5 plis, la Random Forest plusieurs fois plus lentement (les secondes exactes varient d'une exécution à l'autre).

**Réponse à « XGBoost est-il justifié ? »** : pas par sa précision (égalité), mais parce qu'il s'entraîne vite et accepte une perte d'apprentissage sur mesure (section C), ce que la Random Forest de scikit-learn ne permet pas. LightGBM a les mêmes atouts : il est gardé dans la grille de la section C.

<a id="partie-c"></a>
## C. Rendre le modèle plus prudent : où mettre la pénalité ?

**Rappel — score NASA** : pour un écart d = prédiction − vérité, la pénalité vaut exp(d / 10) − 1 si le modèle surestime (d ≥ 0) et exp(−d / 13) − 1 s'il sous-estime ; le score est la somme des pénalités. Exemple : surestimer de 20 cycles coûte e² − 1 ≈ 6,4 ; sous-estimer de 20 cycles coûte e^(20/13) − 1 ≈ 3,7.

### C.1 Fausse piste : changer 13 et 10 dans la note ne change aucune prédiction

**Ce qu'on fait** : on multiplie les deux constantes par un facteur k et on recalcule le score des **mêmes** prédictions test de XGBoost Phase 3. **Pourquoi** : l'idée de départ était de « faire varier 10 et 13 » pour rendre le modèle plus prudent ; on montre que cela change la note, pas le modèle. Le test sert ici à une démonstration, pas à un choix.

In [ ]:
# modele Phase 3 entraine sur les 100 moteurs d'entrainement (comme en Phase 3), puis predictions des 100 moteurs test
modele_phase3_demo = construire_fabrique_depuis_config(CONFIG_XGB_PHASE3)()  # la fabrique, appelee avec () : un modele neuf
modele_phase3_demo.fit(X, y)
predictions_test_phase3_demo = np.clip(modele_phase3_demo.predict(X_test_dernier), a_min=0, a_max=None)  # RUL negative -> 0

# les memes predictions, notees avec les constantes 13 x k et 10 x k
tableau_constantes = demontrer_constantes_score_degenerees(y_test_vraie, predictions_test_phase3_demo,
                                                          facteurs=(1, 2, 5, 10, 100, 1000))
tableau_constantes["rmse_test"] = rmse(y_test_vraie, predictions_test_phase3_demo)  # predictions inchangees : meme RMSE sur chaque ligne
tableau_constantes.round(3)

**Comment lire** : les prédictions et le RMSE sont identiques sur toutes les lignes, seule la note change ; plus k est grand, plus le score tend vers 0, et ce serait vrai pour n'importe quel modèle, même mauvais. « Optimiser » k reviendrait à choisir la note qui flatte le plus.

### C.2 La bonne piste : la pénalité dans la perte d'apprentissage

**Notion — perte d'apprentissage** : la quantité que le modèle rend la plus petite possible pendant l'entraînement. La changer change les prédictions ; la note d'évaluation reste le score NASA 13/10. Deux pertes sur mesure existent dans `src/modeles.py` :
- **quadratique asymétrique** : erreur² × α si le modèle surestime, erreur² sinon. Exemple avec α = 2 : surestimer de 10 cycles coûte 200, sous-estimer de 10 cycles coûte 100 ;
- **forme NASA** : la forme du score NASA, avec des constantes h_sous et h_sur réglables (prolongée par une parabole au-delà de 50 cycles d'écart, pour éviter des exponentielles géantes).

**Ce qu'on fait** : une expérience sur une petite grille, mêmes plis : XGBoost (réglages Phase 3) et LightGBM (réglages de base) avec α ∈ {1 ; 1,5 ; 2 ; 3} ; XGBoost avec la perte NASA (13, 10) et (13, 5), à 200 arbres puis à 1 000 et 2 000 arbres au pas 0,1. Aucun modèle n'est choisi ici.
**Pourquoi 1 000 et 2 000 arbres** : avec la perte NASA, chaque arbre ne déplace les prédictions que d'environ « pas × h » cycles (0,05 × 5 = 0,25 cycle pour h = 5) ; 200 arbres peuvent ne pas suffire à finir l'apprentissage.

In [ ]:
reglages_penalite = []  # une entree par reglage : nom lisible, groupe (pour la figure) et configuration

for alpha in [1.0, 1.5, 2.0, 3.0]:  # XGBoost, perte quadratique asymetrique, reglages Phase 3
    reglage = {}
    reglage["reglage"] = f"XGBoost quadratique alpha={alpha:g}"  # :g affiche 1.0 comme "1" et 1.5 comme "1.5"
    reglage["groupe"] = "XGBoost quadratique asymétrique"
    reglage["config"] = {"famille": "xgboost", "perte": "quadratique_asymetrique",
                         "parametres_perte": {"alpha": alpha}, "hyperparametres": {"n_jobs": 4}}
    reglages_penalite.append(reglage)

for alpha in [1.0, 1.5, 2.0, 3.0]:  # LightGBM, perte quadratique asymetrique, reglages de base
    reglage = {}
    reglage["reglage"] = f"LightGBM quadratique alpha={alpha:g}"
    reglage["groupe"] = "LightGBM quadratique asymétrique"
    reglage["config"] = {"famille": "lightgbm", "perte": "quadratique_asymetrique",
                         "parametres_perte": {"alpha": alpha}, "hyperparametres": {"n_jobs": 4}}
    reglages_penalite.append(reglage)

for h_sous, h_sur in [(13.0, 10.0), (13.0, 5.0)]:  # XGBoost, perte de forme NASA (h_sur = 5 : plus prudent)
    for n_arbres, pas in [(200, 0.05), (1000, 0.1), (2000, 0.1)]:  # reglages Phase 3, puis beaucoup plus d'arbres
        reglage = {}
        reglage["reglage"] = f"XGBoost NASA ({h_sous:g},{h_sur:g}) {n_arbres} arbres pas {pas:g}"
        if n_arbres == 200:
            reglage["groupe"] = "XGBoost perte NASA, 200 arbres"
        else:
            reglage["groupe"] = "XGBoost perte NASA, 1000-2000 arbres"
        reglage["config"] = {"famille": "xgboost", "perte": "nasa",
                             "parametres_perte": {"h_sous": h_sous, "h_sur": h_sur},
                             "hyperparametres": {"n_estimators": n_arbres, "learning_rate": pas, "n_jobs": 4}}
        reglages_penalite.append(reglage)

resultats_cv_penalite = {}  # nom du reglage -> resultats de la validation croisee (memes 5 plis)
for reglage in reglages_penalite:
    fabrique = construire_fabrique_depuis_config(reglage["config"])
    debut = time.perf_counter()
    resultats_cv_penalite[reglage["reglage"]] = evaluer_en_validation_croisee(
        fabrique, X, y, groups, plis, seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE
    )
    print(f"{reglage['reglage']:45s} ({time.perf_counter() - debut:5.1f} s)")  # nom, puis duree des 5 plis

**Tableau de la grille** (la première ligne est la référence de la section B). **Comment lire** : quand α augmente, `surestimes_oof` doit baisser ; `rmse_cv_moyen`, `fausses_alertes_oof` et `biais_oof` montrent le prix payé.

In [ ]:
lignes = []
# premiere ligne : XGBoost Phase 3 de la section B, comme reference
ligne_reference = resumer_cv("XGBoost Phase 3 (perte native)", resultats_cv_base["XGBoost Phase 3"], NB_MOTEURS_TRAIN)
ligne_reference["groupe"] = "référence"
lignes.append(ligne_reference)
for reglage in reglages_penalite:  # puis une ligne par reglage de la grille
    ligne = resumer_cv(reglage["reglage"], resultats_cv_penalite[reglage["reglage"]], NB_MOTEURS_TRAIN)
    ligne["groupe"] = reglage["groupe"]  # sert a la figure C.3
    lignes.append(ligne)
tableau_penalite = pd.DataFrame(lignes).set_index("modele")

# colonnes montrees (sans MAE ni temps de prediction, pour garder le tableau lisible)
colonnes_affichees = ["rmse_cv_moyen", "rmse_cv_ecart_type", "nasa_cv_moyen", "surestimes_oof",
                      "alertes_manquees_oof", "fausses_alertes_oof", "biais_oof", "temps_entrainement_5_plis_s"]
tableau_penalite[colonnes_affichees].round(2)

**Diagnostic de convergence de la perte NASA** : si les chiffres bougent peu entre 1 000 et 2 000 arbres, l'apprentissage a convergé ; s'ils bougent beaucoup entre 200 et 1 000 arbres, les 200 arbres de la Phase 3 n'avaient pas fini d'apprendre.

In [ ]:
for h_sous, h_sur in [(13, 10), (13, 5)]:  # les deux pertes NASA de la grille
    print(f"Perte NASA ({h_sous},{h_sur}) :")
    for n_arbres, pas in [(200, 0.05), (1000, 0.1), (2000, 0.1)]:
        nom_reglage = f"XGBoost NASA ({h_sous:g},{h_sur:g}) {n_arbres} arbres pas {pas:g}"  # meme nom qu'en C.2
        resultats = resultats_cv_penalite[nom_reglage]
        print(f"  {n_arbres:5d} arbres, pas {pas:4.2f} : RMSE CV = {resultats['rmse_moyen']:.2f}, "
              f"NASA CV = {resultats['nasa_moyen']:8.1f}, surestimes OOF > 20 = {resultats['surestimes_oof']:4d}, "
              f"biais OOF = {resultats['biais_oof']:+.2f}")

### C.3 Figure : le compromis précision / prudence

**Comment lire** : un point par réglage. Horizontalement le RMSE CV (à gauche = plus précis) ; verticalement les surestimés OOF > 20 (panneau de gauche) ou le score NASA CV (panneau de droite), en bas = plus prudent. Les points d'une même famille sont reliés par α croissant ; l'étoile noire est XGBoost Phase 3.

In [ ]:
# une ligne par groupe : (nom, couleur, marqueur, style de ligne, decalage de l'etiquette en points, alignement)
# les etiquettes XGBoost vont a gauche des points, celles de LightGBM a droite, pour ne pas se chevaucher
groupes_figure = [
    ("XGBoost quadratique asymétrique", COULEUR_1, "o", "-", (-7, -11), "right"),
    ("LightGBM quadratique asymétrique", COULEUR_2, "s", "-", (7, 3), "left"),
    ("XGBoost perte NASA, 200 arbres", COULEUR_3, "D", "none", (7, -12), "left"),
    ("XGBoost perte NASA, 1000-2000 arbres", COULEUR_3, "^", "none", (7, -10), "left"),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, colonne_y, titre_y in [(axes[0], "surestimes_oof", "surestimés > 20 cycles (lignes OOF)"),
                                (axes[1], "nasa_cv_moyen", "score NASA CV moyen (13/10)")]:  # un panneau a la fois
    for groupe, couleur, marqueur, style_ligne, decalage, alignement in groupes_figure:
        sous_tableau = tableau_penalite[tableau_penalite["groupe"] == groupe]  # les reglages de ce groupe
        ax.plot(sous_tableau["rmse_cv_moyen"], sous_tableau[colonne_y], marker=marqueur, linestyle=style_ligne,
                color=couleur, markersize=8, linewidth=2, label=groupe)
        for nom_reglage in sous_tableau.index:  # etiquette courte a cote de chaque point
            # on retire la partie du nom deja donnee par la legende
            etiquette = nom_reglage.replace("XGBoost quadratique ", "").replace("LightGBM quadratique ", "")
            etiquette = etiquette.replace("XGBoost NASA ", "").replace(" arbres pas ", " arb. pas ")
            decalage_point = decalage
            if "2000 arbres" in nom_reglage:  # 1000 et 2000 arbres sont tres proches : etiquette au-dessus
                decalage_point = (7, 4)
            ax.annotate(etiquette, (sous_tableau.loc[nom_reglage, "rmse_cv_moyen"], sous_tableau.loc[nom_reglage, colonne_y]),
                        textcoords="offset points", xytext=decalage_point, ha=alignement, fontsize=7, color=COULEUR_TEXTE)
    reference = tableau_penalite.loc["XGBoost Phase 3 (perte native)"]  # l'etoile noire
    ax.scatter(reference["rmse_cv_moyen"], reference[colonne_y], marker="*", s=250, color="black", zorder=4,
               label="XGBoost Phase 3 (perte native)")
    ax.set_xlim(16.0, 20.6)  # marge a gauche pour les etiquettes XGBoost
    ax.set_xlabel("RMSE CV moyen (cycles) — plus à gauche = plus précis")
    ax.set_ylabel(titre_y + " — plus bas = plus prudent")
axes[0].set_title("Précision vs surestimations dangereuses")
axes[1].set_title("Précision vs score NASA")
poignees, etiquettes = axes[0].get_legend_handles_labels()  # elements de legende du premier panneau
fig.legend(poignees, etiquettes, loc="lower center", ncol=5, fontsize=9, frameon=False)  # legende commune, sous les panneaux
fig.suptitle("Pénaliser la surestimation dans l'apprentissage : pas de gain gratuit (validation croisée, mêmes 5 plis)")
plt.tight_layout(rect=(0, 0.06, 1, 1))  # on laisse la place de la legende en bas
plt.show()

### C.4 Conclusion de la section C

1. **Changer 13 et 10 dans la note ne change rien au modèle** : RMSE test 19,70 pour tous les k, alors que le score passe de 852,9 (k = 1) à 0,13 (k = 1 000).
2. **La pénalité dans l'apprentissage fonctionne, avec un prix.** Quand α passe de 1 à 3, les surestimés OOF > 20 baissent de 2 386 à 1 590 (−33 %) pour XGBoost et de 2 316 à 1 675 (−28 %) pour LightGBM ; en échange (XGBoost), le RMSE CV monte de 16,86 à 17,68, les fausses alertes de 266 à 435 (+64 %) et le biais passe de −0,2 à −4,2 cycles.
3. **α = 1 redonne la Phase 3** (RMSE CV 16,86 contre 16,87) : la perte sur mesure est bien branchée.
4. **Perte NASA : attention à la convergence.** À 200 arbres, (13, 10) donne le meilleur compromis de la grille (RMSE 16,85 ; score NASA 22 235, soit −25 % ; surestimés 1 806, soit −24 %), mais ce modèle n'a pas fini d'apprendre (biais −2,7) : à 1 000 et 2 000 arbres, chiffres proches entre eux, il devient moins bon que la Phase 3 sur les deux critères (RMSE 17,51 ; score NASA 31 704). Le gain à 200 arbres vient donc de l'apprentissage inachevé, pas de la perte seule.
5. **Conséquence** : pénalité, nombre d'arbres et pas devraient être réglés **ensemble**. Ce notebook ne fait pas cette recherche et ne choisit aucun nouveau modèle : la recherche d'hyperparamètres de la Phase 4 est dans `notebooks/04d_hyperparametres.ipynb` (travail de Matisse, branche `main`).

<a id="partie-d"></a>
## D. Évaluation sur le test officiel (une seule fois)

**Ce qu'on fait** : les trois ensembles d'arbres de la section B sont entraînés sur les 100 moteurs d'entraînement, puis évalués sur le dernier cycle de chaque moteur test, contre la RUL **brute** de `RUL_FD001.txt` (prédictions négatives ramenées à 0). On ajoute le RMSE contre la RUL vraie plafonnée à 125, ce que le modèle pouvait réellement apprendre.
**Contrôle de reproduction** : XGBoost Phase 3 doit redonner RMSE 19,70, score NASA 852,9 et 9 surestimés.

In [ ]:
configs_finales = {}  # nom -> configuration des modeles evalues sur le test (ceux de la section B)
configs_finales["XGBoost Phase 3"] = CONFIG_XGB_PHASE3
configs_finales["Random Forest"] = CONFIG_RANDOM_FOREST
configs_finales["LightGBM"] = CONFIG_LIGHTGBM_BASE

moteurs_test_ordre = contexte_test["moteurs_test"]  # numeros des moteurs test, meme ordre que y_test_vraie
resultats_test = {}  # nom -> dictionnaire renvoye par evaluer_sur_test
for nom_modele in configs_finales:
    modele_final = construire_fabrique_depuis_config(configs_finales[nom_modele])()  # modele neuf
    modele_final.fit(X, y)  # entraine sur les 100 moteurs d'entrainement
    resultats_test[nom_modele] = evaluer_sur_test(modele_final, X_test_dernier, y_test_vraie, plafond=PLAFOND_RUL,
                                                  seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE)

# controle de reproduction du test de la Phase 3 (le notebook s'arrete si un chiffre change)
assert round(resultats_test["XGBoost Phase 3"]["rmse"], 2) == 19.70
assert round(resultats_test["XGBoost Phase 3"]["score_nasa"], 1) == 852.9
assert resultats_test["XGBoost Phase 3"]["nb_surestimes"] == 9

lignes = []
for nom_modele in resultats_test:
    lignes.append(resumer_test(nom_modele, resultats_test[nom_modele]))
tableau_test = pd.DataFrame(lignes).set_index("modele")
tableau_test.round(2)

### D.1 Test apparié sur les 100 moteurs test

**Ce qu'on fait** : erreur absolue de chaque moteur test, Random Forest et LightGBM contre XGBoost Phase 3 (mêmes 100 moteurs). **Comment lire** : différence moyenne négative = le modèle se trompe moins ; avec 100 points seulement, il faut un écart net pour conclure.

In [ ]:
erreurs_absolues_phase3 = np.abs(resultats_test["XGBoost Phase 3"]["predictions"] - y_test_vraie)  # |prediction - verite|, par moteur

lignes = []
for nom_modele in resultats_test:
    if nom_modele == "XGBoost Phase 3":
        continue  # la reference ne se compare pas a elle-meme
    erreurs_absolues_modele = np.abs(resultats_test[nom_modele]["predictions"] - y_test_vraie)
    comparaison = comparer_apparie(erreurs_absolues_modele, erreurs_absolues_phase3)
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["n_moteurs_test"] = comparaison["n"]
    ligne["difference_moyenne_erreur_absolue"] = comparaison["difference_moyenne"]  # negatif = modele meilleur
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(erreurs_absolues_modele < erreurs_absolues_phase3))
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    lignes.append(ligne)
tableau_tests_test = pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)")
tableau_tests_test.round(4)

### D.2 Le score NASA du test, moteur par moteur

**Pourquoi** : le score NASA est une somme d'exponentielles, donc une seule grosse surestimation peut en faire l'essentiel. **Ce qu'on fait** : (1) on compare la pénalité NASA de chaque moteur (test apparié) ; (2) on liste les 5 moteurs qui pèsent le plus dans le score de la Phase 3 ; (3) on recalcule le score total **sans** le moteur le plus pénalisé.

In [ ]:
penalites_test = {}  # nom -> penalite NASA de chacun des 100 moteurs test (un point par moteur)
for nom_modele in resultats_test:
    # metriques_par_moteur renvoie un tableau ; on garde sa colonne "nasa" sous forme de tableau numpy
    penalites_test[nom_modele] = metriques_par_moteur(y_test_vraie, resultats_test[nom_modele]["predictions"],
                                                      moteurs_test_ordre)["nasa"].to_numpy()

# (1) test apparie sur la penalite NASA par moteur
lignes = []
for nom_modele in resultats_test:
    if nom_modele == "XGBoost Phase 3":
        continue
    comparaison = comparer_apparie(penalites_test[nom_modele], penalites_test["XGBoost Phase 3"])
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["difference_moyenne_penalite_nasa"] = comparaison["difference_moyenne"]
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(penalites_test[nom_modele] < penalites_test["XGBoost Phase 3"]))
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    lignes.append(ligne)
print("Penalite NASA par moteur test, tests apparies contre XGBoost Phase 3 :")
print(pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)").round(4).to_string())

# (2) les 5 moteurs qui pesent le plus dans le score NASA de la Phase 3, et leur penalite avec chaque modele
ordre = np.argsort(-penalites_test["XGBoost Phase 3"])  # positions triees du plus penalise au moins penalise
lignes = []
for position in ordre[:5]:  # les 5 premieres positions
    ligne = {}
    ligne["unit_number"] = moteurs_test_ordre[position]
    ligne["rul_vraie"] = y_test_vraie[position]
    for nom_modele in resultats_test:
        ligne[nom_modele] = penalites_test[nom_modele][position]
    lignes.append(ligne)
print("\nLes 5 moteurs les plus penalises par la Phase 3 (penalite NASA de chaque modele) :")
print(pd.DataFrame(lignes).set_index("unit_number").round(1).to_string())

# (3) score total avec et sans le moteur le plus penalise par la Phase 3
position_pire = ordre[0]
print(f"\nScore NASA test total, avec et sans le moteur {moteurs_test_ordre[position_pire]} :")
for nom_modele in resultats_test:
    total = float(np.sum(penalites_test[nom_modele]))  # le score NASA du test = somme des 100 penalites
    sans_le_pire = total - penalites_test[nom_modele][position_pire]
    print(f"  {nom_modele:40s} : {total:7.1f} avec, {sans_le_pire:7.1f} sans")

### D.3 Figure : prédictions contre vraies RUL du test

**Comment lire** : un panneau par modèle ; la diagonale est la prédiction parfaite, au-dessus = surestimation. Les moteurs surestimés de plus de 20 cycles (au-dessus de la ligne pointillée) sont en triangles orange.

In [ ]:
noms_figure_test = ["Random Forest", "XGBoost Phase 3", "LightGBM"]  # meme ordre que la figure B.3
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), sharex=True, sharey=True)  # memes axes pour comparer les panneaux

# bord des axes : la plus grande RUL (vraie ou predite) des trois modeles, plus 5 cycles de marge
limite = y_test_vraie.max()
for nom_modele in noms_figure_test:
    limite = max(limite, resultats_test[nom_modele]["predictions"].max())
limite = limite + 5

for ax, nom_modele in zip(axes, noms_figure_test):  # un panneau par modele
    predictions = resultats_test[nom_modele]["predictions"]
    ecarts = predictions - y_test_vraie
    surestime = ecarts > SEUIL_SURESTIMATION  # Vrai pour les moteurs surestimes de plus de 20 cycles

    ax.scatter(y_test_vraie[~surestime], predictions[~surestime], alpha=0.6, color=COULEUR_1, label="moteurs test")  # ~ : "non"
    ax.scatter(y_test_vraie[surestime], predictions[surestime], alpha=0.9, color=COULEUR_2, marker="^", s=60,
               label=f"surestimés > {SEUIL_SURESTIMATION} cycles ({int(surestime.sum())})")
    ax.plot([0, limite], [0, limite], linestyle="--", color="black", label="prédiction = vérité")
    ax.plot([0, limite], [SEUIL_SURESTIMATION, limite + SEUIL_SURESTIMATION], linestyle=":", color=COULEUR_TEXTE,
            label=f"vérité + {SEUIL_SURESTIMATION}")
    resultats = resultats_test[nom_modele]
    ax.set_title(f"{nom_modele}\nRMSE={resultats['rmse']:.2f}, score NASA={resultats['score_nasa']:.1f}, "
                 f"surestimés>20={resultats['nb_surestimes']}/100", fontsize=10)
    ax.set_xlabel("RUL vraie (RUL_FD001.txt)")
    ax.set_xlim(0, limite)
    ax.set_ylim(0, limite)
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("RUL prédite (modèle entraîné sur les 100 moteurs)")
fig.suptitle("Test officiel FD001 — trois modèles à réglages de base, évalués une seule fois")
plt.tight_layout()
plt.show()

### D.4 Conclusion de la section D

- **Contrôle** : XGBoost Phase 3 redonne exactement RMSE 19,70, score NASA 852,9 et 9 surestimés.
- **Les trois modèles sont proches** : RMSE 19,52 (Random Forest), 19,70 (XGBoost), 20,45 (LightGBM) ; 7 alertes manquées et 1 fausse alerte pour chacun.
- **Aucune différence significative avec XGBoost Phase 3** : erreur absolue par moteur p = 0,97 / 0,76 (Random Forest) et 0,08 / 0,14 (LightGBM) ; pénalité NASA par moteur p = 0,25 / 0,59 et 0,66 / 0,08.
- **Un seul moteur pèse lourd** : le moteur 37 (RUL vraie 21) compte pour 223,6 dans le score NASA de la Phase 3, plus du quart des 852,9. Sans lui : 629,3 (XGBoost), 610,2 (Random Forest), 770,5 (LightGBM).

**Lecture honnête** : le test confirme l'égalité de la section B mais, avec 100 points, il n'a pas la puissance statistique pour départager les modèles.

<a id="partie-e"></a>
## E. Les moteurs surestimés de plus de 20 cycles (XGBoost Phase 3)

**Ce qu'on fait** : on liste les moteurs test que XGBoost Phase 3 surestime de plus de 20 cycles, avec le nombre de cycles **observés** dans `test_FD001.txt` (la longueur de la trajectoire dont dispose le modèle). **Pourquoi** : comprendre les erreurs les plus dangereuses. Cette analyse ne sert à choisir aucun modèle.

In [ ]:
# nombre de cycles observes pour chaque moteur test (trajectoire tronquee), dans l'ordre des moteurs
nb_cycles_par_moteur = contexte_test["test"].groupby("unit_number").size()  # nombre de lignes de chaque moteur
moteurs_test = contexte_test["moteurs_test"]  # numeros de moteur, meme ordre que y_test_vraie
nb_cycles_observes = nb_cycles_par_moteur.loc[moteurs_test].to_numpy()  # remis dans l'ordre de moteurs_test

# un tableau des 100 moteurs test pour XGBoost Phase 3
predictions_phase3 = resultats_test["XGBoost Phase 3"]["predictions"]
tableau_moteurs = pd.DataFrame({
    "unit_number": moteurs_test,
    "rul_vraie": y_test_vraie,
    "rul_predite": predictions_phase3,
    "ecart": predictions_phase3 - y_test_vraie,  # positif = surestimation
    "nb_cycles_observes": nb_cycles_observes,
})
tableau_moteurs["surestime_20"] = tableau_moteurs["ecart"] > SEUIL_SURESTIMATION  # Vrai si surestime de plus de 20 cycles

print(f"=== XGBoost Phase 3 : {int(tableau_moteurs['surestime_20'].sum())} moteurs surestimes de plus de 20 cycles ===")
# seulement les moteurs surestimes, du plus grand ecart au plus petit
surestimes_tries = tableau_moteurs[tableau_moteurs["surestime_20"]].sort_values("ecart", ascending=False)
print(surestimes_tries.round(1).to_string(index=False))

### E.1 Tester des hypothèses plutôt que les affirmer

- **H1** : les moteurs surestimés ont des trajectoires observées **plus courtes** (le modèle a vu peu de cycles, le moteur « a l'air » neuf) ;
- **H2** : les moteurs surestimés ont des RUL vraies **différentes** des autres.

**Notion — test de Mann-Whitney** : il compare deux groupes indépendants (surestimés contre autres) par les rangs, sans supposer de loi normale : « les valeurs d'un groupe ont-elles tendance à être plus grandes ? ». **Limite** : 9 moteurs surestimés seulement ; une p-valeur > 0,05 ne prouve pas l'absence d'effet. En complément, une corrélation de Spearman (ρ, entre −1 et +1 : les deux grandeurs montent-elles ensemble ?) sur les **100 moteurs**, entre l'écart signé et chaque variable.

In [ ]:
groupe_surestimes = tableau_moteurs[tableau_moteurs["surestime_20"]]  # les moteurs surestimes de plus de 20 cycles
groupe_autres = tableau_moteurs[~tableau_moteurs["surestime_20"]]  # tous les autres (~ : "non")

lignes = []
for variable in ["nb_cycles_observes", "rul_vraie"]:  # H1 puis H2
    test_mw = stats.mannwhitneyu(groupe_surestimes[variable], groupe_autres[variable], alternative="two-sided")
    correlation = stats.spearmanr(tableau_moteurs["ecart"], tableau_moteurs[variable])  # sur les 100 moteurs
    ligne = {}
    ligne["variable"] = variable
    ligne["n_surestimes"] = len(groupe_surestimes)
    ligne["n_autres"] = len(groupe_autres)
    ligne["mediane_surestimes"] = groupe_surestimes[variable].median()
    ligne["mediane_autres"] = groupe_autres[variable].median()
    ligne["p_valeur_mann_whitney"] = test_mw.pvalue
    ligne["rho_spearman_ecart_100_moteurs"] = correlation.statistic
    ligne["p_valeur_spearman"] = correlation.pvalue
    lignes.append(ligne)
tableau_hypotheses = pd.DataFrame(lignes).set_index("variable")
tableau_hypotheses.round(4)

### E.2 Figure : écart de prédiction selon la trajectoire observée et selon la RUL vraie

**Comment lire** : un point par moteur test ; au-dessus de la ligne pointillée = surestimé de plus de 20 cycles (triangles orange), sous la ligne noire = sous-estimé.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)  # meme axe vertical (l'ecart) pour les deux panneaux

for ax, variable, titre_x in [(axes[0], "nb_cycles_observes", "nombre de cycles observés dans test_FD001"),
                               (axes[1], "rul_vraie", "RUL vraie (RUL_FD001.txt)")]:  # H1 a gauche, H2 a droite
    ax.scatter(groupe_autres[variable], groupe_autres["ecart"], color=COULEUR_1, marker="o", s=35, alpha=0.7,
               label="autres moteurs")
    ax.scatter(groupe_surestimes[variable], groupe_surestimes["ecart"], color=COULEUR_2, marker="^", s=60, alpha=0.9,
               label=f"surestimés > {SEUIL_SURESTIMATION} cycles")
    ax.axhline(0, color="black", linewidth=1)  # ecart nul : prediction parfaite
    ax.axhline(SEUIL_SURESTIMATION, color=COULEUR_TEXTE, linestyle=":", label=f"surestimation de {SEUIL_SURESTIMATION} cycles")
    ax.set_xlabel(titre_x)
axes[0].set_ylabel("écart = RUL prédite − RUL vraie (cycles) ; > 0 = surestimation")
axes[0].set_title("Écart vs longueur de la trajectoire observée")
axes[1].set_title("Écart vs RUL vraie")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Test officiel FD001, XGBoost Phase 3 — d'où viennent les surestimations ?")
plt.tight_layout()
plt.show()

### E.3 Conclusion de la section E

- **9 moteurs surestimés de plus de 20 cycles**, de +22,6 (moteur 2) à +54,1 cycles (moteur 37).
- **H1 (trajectoires plus courtes) : non démontrée.** Médiane de 121 cycles observés contre 137, mais Mann-Whitney p = 0,07 ; sur les 100 moteurs, ρ = −0,19 (p = 0,06) : au mieux une faible tendance.
- **H2 (RUL vraie différente) : non démontrée entre les deux groupes** (médiane 63 contre 90 ; p = 0,12). La corrélation entre écart et RUL vraie sur 100 moteurs est nette (ρ = −0,34, p < 0,001) mais en partie **mécanique** : la cible apprise est plafonnée à 125, donc un moteur dont la RUL vraie dépasse largement 125 est presque forcément sous-estimé.
- **Les plus dangereux** : les moteurs 37, 18 et 41 (RUL vraie 21, 28 et 18) sont prédits à 75, 62 et 50 cycles, au-dessus du seuil de 30 : ce sont des alertes manquées, sur des trajectoires de longueur ordinaire (121 à 133 cycles). **Piste non testée** : examiner leurs capteurs.

<a id="partie-f"></a>
## F. Améliorations testées (FD001, validation croisée)

Les sections A à E comparent des modèles **à réglages de base**, avec les 46 variables de la Phase 3. Ici, on essaie deux idées pour faire mieux. Une idée n'est gardée que si elle passe une **règle écrite avant de voir les résultats**, jugée sur la validation croisée seulement (les 5 plis de A.4).

| Idée | Modèles concernés | D'où elle vient |
|---|---|---|
| 1. Ajouter `ecartinit_20` : l'écart de chaque capteur à son propre état de départ | les 5 modèles qui apprennent (Random Forest, XGBoost, LightGBM, LSTM, CNN 1D) | notebook `04e_feature_engineering` de Matisse |
| 2. Donner aux réseaux 50 cycles d'historique au lieu de 30 | LSTM et CNN 1D | articles DAST (fenêtres de 40 à 60 cycles) et STAGNN (50 cycles), section H |

**Qui calcule quoi** : les entraînements sont longs, ils sont faits par deux scripts (`scripts/generer_predictions_arbres.py` pour les arbres, `scripts/entrainer_deep_learning.py` pour les réseaux) qui écrivent leurs résultats dans `sorties/resultats/`. Ce notebook **relit** ces résultats : il ne réentraîne aucun des modèles de cette section. Le test officiel n'a été calculé qu'après la décision, une seule fois par version évaluée (F.7) : la version retenue de chaque modèle et, pour les arbres, la version de base (le modèle final publié avant cette étude) ; un candidat non retenu n'est jamais évalué sur le test. Pour XGBoost, la version à 31 variables avait déjà été évaluée une fois sur le test par Matisse (04e, après son choix en validation croisée) : le script redonne exactement son chiffre.

### F.1 Lecture des résultats des scripts

**Ce qu'on fait** : on vérifie que tous les fichiers produits par les scripts sont présents. S'il en manque un, la cellule s'arrête et affiche la commande à lancer depuis la racine du dépôt (rien n'est réentraîné ici). On importe aussi les fonctions utilisées dans les sections F à H.

In [ ]:
# bibliotheques supplementaires pour les sections F a I
import json      # lecture des regles et des decisions enregistrees par les scripts
import os        # chemins de fichiers
import textwrap  # coupe les longs textes en lignes de largeur fixe pour l'affichage

from IPython.display import display  # affiche plusieurs tableaux dans une meme cellule
from sklearn.dummy import DummyRegressor  # baselines constantes (section G)
from sklearn.linear_model import Ridge  # baseline lineaire (section G)
from sklearn.pipeline import Pipeline  # mise a l'echelle puis Ridge, appris ensemble dans chaque pli
from sklearn.preprocessing import StandardScaler  # mise a l'echelle apprise sur le train du pli

from src.contrat_predictions import extraire_dernier_cycle, lire_metadonnees, lire_predictions
from src.evaluation import (
    choisir_seuil_alerte,             # seuil d'alerte choisi sur la validation (section G.5)
    comparer_modeles_par_paires,      # toutes les comparaisons appariees, avec Bonferroni (section G.6)
    erreurs_par_tranche_rul,          # erreur par tranche de RUL vraie (section G.4)
    metriques_classification_alerte,  # matrice de confusion, precision, rappel, F1, AUC (section G.5)
    metriques_regression,             # toutes les metriques de regression d'un ensemble de predictions
    metriques_regression_par_pli,     # les memes, pli par pli, a partir des predictions hors-pli
    resumer_plis,                     # moyenne et ecart-type entre les plis
)
from src.sous_jeux import charger_rul_sous_jeu, preparer_sous_jeu  # preparation par sous-jeu (option ecart_initial)

DOSSIER_RESULTATS = os.path.join("..", "sorties", "resultats")      # resultats des scripts (non versionnes)
DOSSIER_PREDICTIONS = os.path.join("..", "sorties", "predictions")  # predictions au format du contrat
SOUS_JEUX = ["FD001", "FD002", "FD003", "FD004"]

# commandes a lancer depuis la racine du depot si un fichier manque (durees mesurees sur la machine du projet)
COMMANDE_ARBRES = ".venv/bin/python scripts/generer_predictions_arbres.py   (environ 12 minutes)"
COMMANDE_RESEAUX = ".venv/bin/python scripts/entrainer_deep_learning.py --cv-sous-jeux   (environ 1 heure)"

# les 6 modeles retenus : cle des fichiers -> nom affiche dans les tableaux
MODELES_RETENUS = {}
MODELES_RETENUS["reference_naive"] = "Reference naive"
MODELES_RETENUS["random_forest"] = "Random Forest"
MODELES_RETENUS["lightgbm"] = "LightGBM"
MODELES_RETENUS["xgboost_phase3"] = "XGBoost"
MODELES_RETENUS["lstm"] = "LSTM"
MODELES_RETENUS["cnn1d"] = "CNN 1D"
CLES_ARBRES = ["reference_naive", "random_forest", "xgboost_phase3", "lightgbm"]  # produits par le script des arbres
CLES_RESEAUX = ["lstm", "cnn1d"]  # produits par le script des reseaux

# fichier attendu -> commande qui le produit
FICHIERS_ATTENDUS = {}
for nom_fichier in ["05_arbres_choix_features.csv", "05_arbres_resume.csv", "05_arbres_oof.csv.gz",
                    "05_arbres_test_dernier_cycle.csv"]:
    FICHIERS_ATTENDUS[os.path.join(DOSSIER_RESULTATS, nom_fichier)] = COMMANDE_ARBRES
for nom_fichier in ["03b_candidats_fd001.csv", "03b_candidats_fd001.json", "03b_oof_fd001.csv.gz",
                    "03b_predictions_test_fd001.csv", "03b_test_fd001.csv", "03b_deep_learning.csv"]:
    FICHIERS_ATTENDUS[os.path.join(DOSSIER_RESULTATS, nom_fichier)] = COMMANDE_RESEAUX
for sous_jeu in SOUS_JEUX:  # validation croisee 5 plis des reseaux, un fichier par sous-jeu
    FICHIERS_ATTENDUS[os.path.join(DOSSIER_RESULTATS, f"03b_cv_{sous_jeu.lower()}.csv")] = COMMANDE_RESEAUX
for cle in MODELES_RETENUS:  # metadonnees des predictions FD001 (regle de decision, metriques publiees)
    if cle in CLES_RESEAUX:
        FICHIERS_ATTENDUS[os.path.join(DOSSIER_PREDICTIONS, f"FD001__{cle}.json")] = COMMANDE_RESEAUX
    else:
        FICHIERS_ATTENDUS[os.path.join(DOSSIER_PREDICTIONS, f"FD001__{cle}.json")] = COMMANDE_ARBRES

manquants = []  # (fichier, commande) des fichiers absents
for chemin in FICHIERS_ATTENDUS:
    if not os.path.exists(chemin):
        manquants.append((chemin, FICHIERS_ATTENDUS[chemin]))

if len(manquants) > 0:
    print("Fichiers manquants, a produire avec la commande indiquee (depuis la racine du depot) :")
    for chemin, commande in manquants:
        print(f"  {chemin}\n      -> {commande}")
    raise FileNotFoundError(f"{len(manquants)} fichier(s) de resultats manquant(s) : voir la liste ci-dessus.")
print(f"Les {len(FICHIERS_ATTENDUS)} fichiers de resultats attendus sont presents.")

### F.2 Idée 1 — la variable `ecartinit_20` (Matisse, notebook 04e)

**L'idée.** Deux moteurs neufs n'ont pas exactement les mêmes valeurs de capteurs (petites différences de fabrication). La valeur d'un capteur seule dit donc mal où en est un moteur ; ce qui annonce la panne, c'est **de combien il s'est éloigné de son propre état de départ**. Pour chaque capteur lissé x d'un moteur :

ecartinit_20(t) = x(t) − moyenne de x sur les 20 premiers cycles du moteur.

Tant que t < 20, la moyenne porte sur les t cycles déjà vus : on n'utilise **jamais** un cycle futur (vérifié par `test_ecart_etat_initial_causal_et_par_moteur` dans `tests/test_anti_triche.py`).

**Exemple** (avec 3 cycles de référence au lieu de 20, pour la lisibilité) : un capteur vaut 10, 12, 14 puis 20. La référence vaut 10 (cycle 1), puis 11, puis 12, et reste figée à 12 ensuite. L'écart vaut donc 0, 1, 2 puis 8.

**D'où elle vient.** Le notebook `04e_feature_engineering` de Matisse conclut un jeu de **31 variables** : les 15 capteurs lissés, `time_cycles` et les 15 écarts `<capteur>_ecartinit_20`. Pour les arbres, ce jeu **remplace** les 46 variables de base. Pour les réseaux (candidat « A »), on ajoute les 15 écarts (mis à l'échelle entre 0 et 1) aux 15 capteurs et à `time_cycles` / 100 : 31 entrées aussi.

**Ce qu'on attend.** Une baisse nette de l'erreur pour tous les modèles, puisque la variable rend comparables des moteurs qui partent de niveaux différents.

**Règle de décision**, écrite dans les scripts avant les expériences (texte commun aux deux scripts, recopié tel quel) :

> Pour chaque modele, une amelioration est retenue sur FD001 si, sur les memes 5 plis par moteur, (a) le RMSE moyen de validation baisse d'au moins 0,3 cycle (seuil de Matisse, notebook 04d) ET (b) le test de Wilcoxon apparie sur le RMSE par moteur (100 moteurs, prediction hors-pli) donne p < 0,05 en faveur du candidat. Sinon on garde la version de base. Le score NASA et les surestimations de plus de 20 cycles sont rapportes a cote ; si le candidat les degrade nettement, le dire. Les choix faits sur FD001 sont appliques tels quels a FD002-FD004 (le dire comme limite).

Mise en œuvre, fixée elle aussi avant les résultats : la p-valeur de Wilcoxon est **bilatérale** (`comparer_apparie` de `src/evaluation.py`) et « en faveur du candidat » veut dire que la différence moyenne (RMSE candidat − RMSE base, moteur par moteur) est **négative**. La cellule suivante affiche les textes complets tels que les scripts les ont enregistrés.

In [ ]:
# regle des arbres : recopiee par le script dans les metadonnees de chaque fichier de predictions
with open(os.path.join(DOSSIER_PREDICTIONS, "FD001__xgboost_phase3.json"), encoding="utf-8") as fichier:
    metadonnees_xgb_fd001 = json.load(fichier)
# regle des reseaux : enregistree par le script avec ses decisions
with open(os.path.join(DOSSIER_RESULTATS, "03b_candidats_fd001.json"), encoding="utf-8") as fichier:
    decisions_reseaux = json.load(fichier)

print("Regle des modeles a arbres (scripts/generer_predictions_arbres.py) :")
print(textwrap.fill(metadonnees_xgb_fd001["regle_de_decision_features"], width=110))
print("\nRegle des reseaux (scripts/entrainer_deep_learning.py) :")
print(textwrap.fill(decisions_reseaux["regle"], width=110))
print("\nPrecisions des reseaux, fixees avant les resultats :")
print(textwrap.fill(decisions_reseaux["precisions"], width=110))

### F.3 Résultat de l'idée 1 pour les modèles à arbres

**Comment lire** : une ligne par modèle. `rmse_cv` = moyenne ± écart-type des 5 plis (en cycles) ; `(a)` et `(b)` = les deux conditions de la règle ; `moteurs_meilleurs` = nombre de moteurs (sur 100) où le candidat a un RMSE plus petit ; les trois dernières colonnes de chiffres disent si le candidat dégrade la prudence (score NASA de validation, lignes surestimées de plus de 20 cycles, alertes manquées). La référence naïve prédit toujours la même valeur : elle ignore les variables, aucun test n'est possible et elle garde la base.

In [ ]:
def formater_moyenne_ecart(moyenne, ecart_type, decimales=2):
    """
    Ecrit une moyenne et un ecart-type sous la forme "moyenne ± ecart-type".

    Parameters
    ----------
    moyenne : float
        La moyenne.
    ecart_type : float
        L'ecart-type.
    decimales : int, optionnel
        Nombre de decimales affichees (2 par defaut).

    Returns
    -------
    str
        Par exemple "9.36 ± 0.33".
    """
    return f"{moyenne:.{decimales}f} ± {ecart_type:.{decimales}f}"


def p_valeur_lisible(p_valeur):
    """
    Ecrit une p-valeur en notation scientifique courte, ou "sans objet" si elle n'existe pas.

    Parameters
    ----------
    p_valeur : float
        La p-valeur (NaN si aucun test n'a ete possible).

    Returns
    -------
    str
        Par exemple "9.9e-16", ou "sans objet".
    """
    if pd.isna(p_valeur):
        return "sans objet"
    return f"{p_valeur:.1e}"


choix_arbres = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "05_arbres_choix_features.csv"))
choix_fd001 = choix_arbres[choix_arbres["sous_jeu"] == "FD001"].set_index("modele")  # la decision se prend sur FD001

lignes = []
for cle in CLES_ARBRES:
    source = choix_fd001.loc[cle]  # la ligne du script pour ce modele
    ligne = {}
    ligne["modele"] = MODELES_RETENUS[cle]
    ligne["rmse_cv_base_46"] = formater_moyenne_ecart(source["rmse_cv_base"], source["rmse_cv_ecart_type_base"])
    ligne["rmse_cv_candidat_31"] = formater_moyenne_ecart(source["rmse_cv_candidat"], source["rmse_cv_ecart_type_candidat"])
    ligne["baisse"] = round(source["baisse_rmse_cv"], 2)
    ligne["(a) baisse >= 0,3"] = bool(source["condition_a_baisse_rmse"])
    ligne["p_wilcoxon"] = p_valeur_lisible(source["p_wilcoxon_bilaterale"])
    ligne["moteurs_meilleurs"] = int(source["nb_moteurs_candidat_meilleur"])
    ligne["(b) p < 0,05 en faveur"] = bool(source["condition_b_wilcoxon"])
    ligne["nasa_cv base -> candidat"] = f"{source['nasa_cv_base']:.0f} -> {source['nasa_cv_candidat']:.0f}"
    ligne["surestimes_oof base -> candidat"] = f"{source['surestimes_oof_base']} -> {source['surestimes_oof_candidat']}"
    ligne["alertes_manquees_oof base -> candidat"] = (
        f"{source['alertes_manquees_oof_base']} -> {source['alertes_manquees_oof_candidat']}")
    ligne["decision"] = source["jeu_retenu"]
    lignes.append(ligne)
tableau_choix_arbres = pd.DataFrame(lignes).set_index("modele")
tableau_choix_arbres

### F.4 Résultat de l'idée 1 pour les réseaux (candidat A)

**Ce qu'on fait** : même règle, appliquée à chaque réseau séparément. « base » = fenêtre de 30 cycles, 15 capteurs + `time_cycles` / 100 (16 entrées, notebook 03b) ; « A » = la même chose + les 15 écarts `ecartinit_20` (31 entrées). Mêmes colonnes qu'en F.3 ; `compare_a` dit contre quelle version le candidat est jugé.

In [ ]:
candidats_reseaux = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_candidats_fd001.csv"))
NOMS_RESEAUX = {"lstm": "LSTM", "cnn1d": "CNN 1D"}
DESCRIPTION_CANDIDATS = {}  # code du candidat -> description courte
DESCRIPTION_CANDIDATS["base"] = "fenetre 30"
DESCRIPTION_CANDIDATS["A"] = "fenetre 30 + ecartinit_20"
DESCRIPTION_CANDIDATS["B"] = "fenetre 50"
DESCRIPTION_CANDIDATS["AB"] = "fenetre 50 + ecartinit_20"


def texte_ou_tiret(valeur):
    """
    Renvoie la valeur sous forme de texte, ou un tiret si elle est absente (cas de la version de base).

    Parameters
    ----------
    valeur : object
        Une valeur lue dans le fichier (NaN si absente).

    Returns
    -------
    str
        La valeur en texte, ou "-".
    """
    if pd.isna(valeur):
        return "-"
    return str(valeur)


def tableau_candidats_reseaux(candidats, codes_a_montrer):
    """
    Met en forme les candidats des reseaux (fichier 03b_candidats_fd001.csv), une ligne par reseau et candidat.

    Parameters
    ----------
    candidats : pandas.DataFrame
        Le contenu de 03b_candidats_fd001.csv.
    codes_a_montrer : list de str
        Les codes des candidats a afficher ("base", "A", "B", "AB").

    Returns
    -------
    pandas.DataFrame
        Index (reseau, candidat) ; RMSE CV, comparaison, conditions de la regle et indicateurs de prudence.
    """
    lignes = []
    for reseau in ["lstm", "cnn1d"]:  # LSTM d'abord, puis CNN 1D
        for code_candidat in codes_a_montrer:
            selection = candidats[(candidats["type_modele"] == reseau) & (candidats["candidat"] == code_candidat)]
            if len(selection) == 0:
                continue  # candidat non evalue pour ce reseau (A+B seulement si A et B passent la regle)
            source = selection.iloc[0]
            ligne = {}
            ligne["reseau"] = NOMS_RESEAUX[reseau]
            ligne["candidat"] = code_candidat
            ligne["entrees"] = DESCRIPTION_CANDIDATS[code_candidat]
            ligne["nb_entrees"] = int(source["nb_entrees"])
            ligne["rmse_cv"] = formater_moyenne_ecart(source["rmse_cv_moyen"], source["rmse_cv_ecart_type"])
            ligne["compare_a"] = texte_ou_tiret(source["compare_a"])
            if pd.isna(source["gain_rmse"]):
                ligne["baisse"] = "-"
            else:
                ligne["baisse"] = f"{source['gain_rmse']:.2f}"
            if pd.isna(source["p_wilcoxon"]):
                ligne["p_wilcoxon"] = "-"
            else:
                ligne["p_wilcoxon"] = p_valeur_lisible(source["p_wilcoxon"])
            ligne["moteurs_meilleurs"] = texte_ou_tiret(source["nb_moteurs_ameliores"]).replace(".0", "")
            ligne["(a)"] = texte_ou_tiret(source["critere_a"])
            ligne["(b)"] = texte_ou_tiret(source["critere_b"])
            ligne["retenu"] = texte_ou_tiret(source["retenu"])
            ligne["nasa_cv"] = round(source["nasa_cv_moyen"])
            ligne["surestimes_oof"] = int(source["surestimes_oof"])
            ligne["alertes_manquees_oof"] = int(source["alertes_manquees_oof"])
            lignes.append(ligne)
    return pd.DataFrame(lignes).set_index(["reseau", "candidat"])


tableau_candidats_reseaux(candidats_reseaux, ["base", "A"])

### F.5 Idée 2 — une fenêtre de 50 cycles pour les réseaux

**L'idée.** Le LSTM et le CNN 1D lisent les 30 derniers cycles d'un moteur. Avec 50 cycles, ils voient une plus longue partie de la dégradation, donc une tendance plus nette.

**D'où elle vient.** Deux articles récents utilisent des fenêtres plus longues que 30 : DAST (Zhang, Song, Li 2022) prend 40 cycles sur FD001 et FD003, 60 sur FD002 et FD004 ; STAGNN (Huang, He, Sick 2023) prend 50 cycles (références complètes en section H).

**Ce qu'on attend, et le risque.** Une baisse de l'erreur, au prix d'un **remplissage** plus fréquent : quand un moteur a moins de 50 cycles derrière lui, le début de la fenêtre est complété en répétant son premier cycle (notebook 03b, B.2). Sur le train FD001, chaque moteur a au moins 128 cycles : seuls ses 49 premiers cycles sont concernés. Sur le test FD001, 7 moteurs ont moins de 50 cycles observés : leur fenêtre serait remplie au moment même où on les note.

**Règle** : la même qu'en F.2, avec une précision fixée avant les résultats : si A et B passent chacun la règle contre la base, la combinaison A+B (fenêtre 50 + `ecartinit_20`) est comparée, avec la même règle, **au meilleur des deux** ; sinon on garde celui qui passe.

**Comment lire** : mêmes colonnes qu'en F.4. La ligne « AB » est comparée à A (colonne `compare_a`), pas à la base.

In [ ]:
tableau_candidats_reseaux(candidats_reseaux, ["base", "B", "AB"])

**Décision enregistrée par le script**, pour chaque réseau : les candidats qui passent la règle contre la base, puis la version finalement retenue (longueur de fenêtre et nombre de cycles de référence de `ecartinit`).

In [ ]:
for reseau in ["lstm", "cnn1d"]:
    decision = decisions_reseaux["decisions"][reseau]  # bloc de decision du script pour ce reseau
    print(f"{NOMS_RESEAUX[reseau]:7s} : candidats qui passent la regle contre la base = {decision['retenus_contre_base']} ; "
          f"version retenue = {decision['configuration_retenue']} "
          f"(fenetre {decision['longueur_fenetre']} cycles, ecart_initial = {decision['ecart_initial']})")

### F.6 Figure : erreur de validation avant et après l'idée 1

**Comment lire** : pour chaque modèle qui apprend, la barre grise est la version de base, la barre bleue la version avec `ecartinit_20` (celle qui est retenue) ; le trait vertical est l'écart-type entre les 5 plis. Plus bas = mieux. Mêmes 5 plis pour toutes les barres.

In [ ]:
noms_figure = []      # noms des 5 modeles, dans l'ordre des barres
rmse_base = []        # RMSE CV moyen de la version de base
ecart_base = []       # ecart-type entre plis de la version de base
rmse_retenu = []      # RMSE CV moyen de la version avec ecartinit_20
ecart_retenu = []     # ecart-type entre plis de cette version
for cle in ["random_forest", "xgboost_phase3", "lightgbm"]:  # les arbres, lus en F.3
    noms_figure.append(MODELES_RETENUS[cle])
    rmse_base.append(choix_fd001.loc[cle, "rmse_cv_base"])
    ecart_base.append(choix_fd001.loc[cle, "rmse_cv_ecart_type_base"])
    rmse_retenu.append(choix_fd001.loc[cle, "rmse_cv_candidat"])
    ecart_retenu.append(choix_fd001.loc[cle, "rmse_cv_ecart_type_candidat"])
for reseau in ["lstm", "cnn1d"]:  # les reseaux, lus en F.4
    ligne_base = candidats_reseaux[(candidats_reseaux["type_modele"] == reseau) & (candidats_reseaux["candidat"] == "base")].iloc[0]
    ligne_a = candidats_reseaux[(candidats_reseaux["type_modele"] == reseau) & (candidats_reseaux["candidat"] == "A")].iloc[0]
    noms_figure.append(NOMS_RESEAUX[reseau])
    rmse_base.append(ligne_base["rmse_cv_moyen"])
    ecart_base.append(ligne_base["rmse_cv_ecart_type"])
    rmse_retenu.append(ligne_a["rmse_cv_moyen"])
    ecart_retenu.append(ligne_a["rmse_cv_ecart_type"])

positions_f6 = np.arange(len(noms_figure))  # une position par modele
largeur = 0.38  # largeur d'une barre ; les deux barres d'un modele sont cote a cote
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(positions_f6 - largeur / 2, rmse_base, width=largeur, yerr=ecart_base, capsize=4, color="#a9a8a3",
       label="version de base")
ax.bar(positions_f6 + largeur / 2, rmse_retenu, width=largeur, yerr=ecart_retenu, capsize=4, color=COULEUR_1,
       label="avec ecartinit_20 (retenue)")
for position, valeur, ecart in zip(positions_f6, rmse_retenu, ecart_retenu):  # valeur au-dessus de chaque barre bleue
    ax.text(position + largeur / 2, valeur + ecart + 0.4, f"{valeur:.2f}", ha="center", fontsize=9, color=COULEUR_TEXTE)
for position, valeur, ecart in zip(positions_f6, rmse_base, ecart_base):  # et au-dessus de chaque barre grise
    ax.text(position - largeur / 2, valeur + ecart + 0.4, f"{valeur:.2f}", ha="center", fontsize=9, color=COULEUR_TEXTE)
ax.set_xticks(positions_f6)
ax.set_xticklabels(noms_figure)
ax.set_ylabel("RMSE de validation croisée (cycles), moyenne des 5 plis")
ax.set_ylim(0, 24)  # place pour la legende au-dessus des barres
ax.set_title("FD001 — l'écart à l'état initial fait baisser l'erreur de tous les modèles (mêmes 5 plis)")
ax.legend(loc="upper center", ncol=2)
plt.tight_layout()
plt.show()

### F.7 Test officiel des versions retenues, FD001 à FD004

**Ce qu'on fait** : le choix fait sur FD001 est appliqué tel quel à FD002, FD003 et FD004 (c'est une limite : il n'est pas refait par sous-jeu). Chaque modèle retenu est entraîné sur tous les moteurs d'entraînement du sous-jeu, puis évalué **une seule fois** sur le dernier cycle observé de chaque moteur test.

**Comment lire** :
- `rmse_cv` : validation croisée 5 plis (mêmes plis que les arbres pour les réseaux aussi) ; `base` = version de base, `retenu` = version retenue ;
- `test_brute` : RMSE contre la vraie RUL de `RUL_FD00X.txt` ; `test_plafonnee` : RMSE contre cette RUL plafonnée à 125, ce que le modèle pouvait apprendre ;
- pour les arbres, la colonne `base` du test est le modèle final publié avant cette étude, recalculé à l'identique ; pour les réseaux, le test de la version de base n'est plus dans les fichiers (« - ») ;
- `surestimes_test` = moteurs surestimés de plus de 20 cycles ; `alertes_manquees_test` = vraie RUL ≤ 30 mais prédiction > 30.

In [ ]:
resume_arbres = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "05_arbres_resume.csv"))  # arbres : base et retenu
resultats_reseaux = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_deep_learning.csv"))  # reseaux : retenu
cv_reseaux = {}  # sous-jeu -> validation croisee 5 plis des reseaux retenus
for sous_jeu in SOUS_JEUX:
    cv_reseaux[sous_jeu] = pd.read_csv(os.path.join(DOSSIER_RESULTATS, f"03b_cv_{sous_jeu.lower()}.csv")).set_index("modele")

lignes = []
for sous_jeu in SOUS_JEUX:
    for cle in CLES_ARBRES:
        lignes_modele = resume_arbres[(resume_arbres["sous_jeu"] == sous_jeu) & (resume_arbres["modele"] == cle)]
        base = lignes_modele[lignes_modele["jeu_features"] == "base"].iloc[0]  # 46 variables (ou equivalent par sous-jeu)
        retenu = lignes_modele[lignes_modele["retenu"]].iloc[0]  # la version choisie par la regle sur FD001
        ligne = {}
        ligne["sous_jeu"] = sous_jeu
        ligne["modele"] = MODELES_RETENUS[cle]
        ligne["version_retenue"] = retenu["jeu_features"]
        ligne["moteurs_test"] = int(retenu["nb_moteurs_test"])
        ligne["rmse_cv_base"] = base["rmse_cv_moyen"]
        ligne["rmse_cv_retenu"] = retenu["rmse_cv_moyen"]
        ligne["test_brute_base"] = base["rmse_test"]
        ligne["test_brute_retenu"] = retenu["rmse_test"]
        ligne["test_plafonnee_base"] = base["rmse_test_rul_plafonnee"]
        ligne["test_plafonnee_retenu"] = retenu["rmse_test_rul_plafonnee"]
        ligne["nasa_test_retenu"] = retenu["nasa_test"]
        ligne["surestimes_test"] = int(retenu["surestimes_test"])
        ligne["alertes_manquees_test"] = int(retenu["alertes_manquees_test"])
        lignes.append(ligne)
    for cle in CLES_RESEAUX:
        retenu = resultats_reseaux[(resultats_reseaux["sous_jeu"] == sous_jeu) & (resultats_reseaux["modele"] == cle)].iloc[0]
        ligne = {}
        ligne["sous_jeu"] = sous_jeu
        ligne["modele"] = MODELES_RETENUS[cle]
        ligne["version_retenue"] = decisions_reseaux["decisions"][cle]["configuration_retenue"]
        ligne["moteurs_test"] = int(retenu["nb_moteurs_test"])
        if sous_jeu == "FD001":  # la version de base des reseaux n'a ete evaluee en validation croisee que sur FD001
            ligne_base = candidats_reseaux[(candidats_reseaux["type_modele"] == cle) & (candidats_reseaux["candidat"] == "base")].iloc[0]
            ligne["rmse_cv_base"] = ligne_base["rmse_cv_moyen"]
        else:
            ligne["rmse_cv_base"] = np.nan
        ligne["rmse_cv_retenu"] = cv_reseaux[sous_jeu].loc[cle, "rmse_cv_moyen"]
        ligne["test_brute_base"] = np.nan  # test de la version de base des reseaux : plus dans les fichiers
        ligne["test_brute_retenu"] = retenu["rmse_test"]
        ligne["test_plafonnee_base"] = np.nan
        ligne["test_plafonnee_retenu"] = retenu["rmse_test_rul_plafonnee"]
        ligne["nasa_test_retenu"] = retenu["nasa_test"]
        ligne["surestimes_test"] = int(retenu["surestimes_test"])
        ligne["alertes_manquees_test"] = int(retenu["alertes_manquees_test"])
        lignes.append(ligne)
tableau_test_sous_jeux = pd.DataFrame(lignes).set_index(["sous_jeu", "modele"])
tableau_test_sous_jeux.round(2).fillna("-")  # "-" : valeur absente des fichiers

### F.8 Ce qu'il faut retenir de la section F

- **Idée 1 retenue pour les 5 modèles qui apprennent.** RMSE de validation FD001 : Random Forest 16,96 → 10,19 ; XGBoost 16,87 → 9,36 ; LightGBM 16,85 → 9,35 ; LSTM 13,70 → 8,94 ; CNN 1D 16,96 → 10,05. Les deux conditions passent partout : baisse de 4,76 à 7,51 cycles, Wilcoxon p ≤ 2,2e-13, candidat meilleur sur 83 à 87 moteurs sur 100. La référence naïve garde la base (elle n'utilise pas les variables).
- **La prudence ne se dégrade pas, elle s'améliore** : lignes hors-pli surestimées de plus de 20 cycles 2 384 → 754 (XGBoost) et 1 503 → 799 (LSTM) ; alertes manquées hors-pli 552 → 254 (XGBoost) et 298 → 154 (LSTM).
- **Idée 2 non retenue.** LSTM : la fenêtre de 50 cycles passe la règle contre la base (13,70 → 11,32), mais la combinaison A+B (9,32) ne bat pas A (8,94 ; p = 0,063) : A est gardé. CNN 1D : la fenêtre de 50 cycles baisse le RMSE de 0,89 cycle, sans effet significatif (p = 0,41) : rejetée.
- **Test officiel FD001**, calculé après la décision (RMSE contre la RUL brute / plafonnée) : LSTM 12,29 / 11,37 ; CNN 1D 12,74 / 12,38 ; XGBoost 13,10 / 11,88 (19,70 / 18,53 en version de base) ; LightGBM 13,40 / 12,12 ; Random Forest 14,27 / 13,03.
- **FD002 à FD004** : pour les trois modèles à arbres, la version retenue baisse le RMSE test brut sur les trois sous-jeux (par exemple XGBoost FD004 : 28,24 → 24,76). Sur FD002 et FD004, le RMSE brut des modèles retenus (22,57 à 29,25) reste nettement au-dessus du RMSE plafonné (10,95 à 22,51) : beaucoup de moteurs test y ont une vraie RUL au-delà de 125 (H.1).
- **Alerte : le CNN 1D sur FD004.** Score NASA test 446 374 (XGBoost : 3 721 ; LSTM : 3 310), 55 moteurs surestimés de plus de 20 cycles sur 248, 23 alertes manquées. Le RMSE de validation croisée ne laissait pas prévoir cet écart (11,60 contre 11,12). Ce constat vient du test : il ne change pas le choix (règle 1 de A.1) et il est rapporté comme limite ; la démo propose ce modèle, ses prédictions FD004 sont à lire avec prudence.
- **Limites** : choix fait sur FD001 seulement ; hyperparamètres non réoptimisés pour les nouvelles entrées ; un seul entraînement par modèle (3 graines pour les réseaux sur FD001, section H).

**Contrôle des chiffres de la section F** : la cellule suivante vérifie que les chiffres cités ci-dessus (et dans les cellules F.2 et F.5) sont ceux des fichiers relus ; elle s'arrête sur une erreur si un résultat change.

In [ ]:
# le texte commun de la regle est bien le meme dans les deux scripts
assert metadonnees_xgb_fd001["regle_de_decision_features"].startswith(decisions_reseaux["regle"])

# F.5 : longueur des trajectoires FD001 (remplissage d'une fenetre de 50 cycles)
longueurs_train = groups.value_counts()                                # cycles par moteur d'entrainement (A.3)
longueurs_test = contexte_test["test"].groupby("unit_number").size()   # cycles observes par moteur test (A.3)
assert longueurs_train.min() == 128
assert int(np.sum(longueurs_test < 50)) == 7

# idee 1 : RMSE CV base -> retenu, et prudence
rmse_cv_attendus = {"random_forest": (16.96, 10.19), "xgboost_phase3": (16.87, 9.36), "lightgbm": (16.85, 9.35)}
for cle in rmse_cv_attendus:
    assert round(choix_fd001.loc[cle, "rmse_cv_base"], 2) == rmse_cv_attendus[cle][0]
    assert round(choix_fd001.loc[cle, "rmse_cv_candidat"], 2) == rmse_cv_attendus[cle][1]
    assert choix_fd001.loc[cle, "jeu_retenu"] == "ecart_initial_20"
assert choix_fd001.loc["reference_naive", "jeu_retenu"] == "base"
candidats_indexes = candidats_reseaux.set_index(["type_modele", "candidat"])
assert round(candidats_indexes.loc[("lstm", "base"), "rmse_cv_moyen"], 2) == 13.70
assert round(candidats_indexes.loc[("lstm", "A"), "rmse_cv_moyen"], 2) == 8.94
assert round(candidats_indexes.loc[("cnn1d", "base"), "rmse_cv_moyen"], 2) == 16.96
assert round(candidats_indexes.loc[("cnn1d", "A"), "rmse_cv_moyen"], 2) == 10.05
baisses = [choix_fd001.loc["random_forest", "baisse_rmse_cv"], choix_fd001.loc["xgboost_phase3", "baisse_rmse_cv"],
           choix_fd001.loc["lightgbm", "baisse_rmse_cv"], candidats_indexes.loc[("lstm", "A"), "gain_rmse"],
           candidats_indexes.loc[("cnn1d", "A"), "gain_rmse"]]
assert round(min(baisses), 2) == 4.76 and round(max(baisses), 2) == 7.51
p_valeurs = [choix_fd001.loc["random_forest", "p_wilcoxon_bilaterale"], choix_fd001.loc["xgboost_phase3", "p_wilcoxon_bilaterale"],
             choix_fd001.loc["lightgbm", "p_wilcoxon_bilaterale"], candidats_indexes.loc[("lstm", "A"), "p_wilcoxon"],
             candidats_indexes.loc[("cnn1d", "A"), "p_wilcoxon"]]
assert max(p_valeurs) <= 2.2e-13 + 0.05e-13  # arrondi a 2,2e-13
moteurs_meilleurs = [choix_fd001.loc["random_forest", "nb_moteurs_candidat_meilleur"],
                     choix_fd001.loc["xgboost_phase3", "nb_moteurs_candidat_meilleur"],
                     choix_fd001.loc["lightgbm", "nb_moteurs_candidat_meilleur"],
                     candidats_indexes.loc[("lstm", "A"), "nb_moteurs_ameliores"],
                     candidats_indexes.loc[("cnn1d", "A"), "nb_moteurs_ameliores"]]
assert min(moteurs_meilleurs) == 83 and max(moteurs_meilleurs) == 87
assert choix_fd001.loc["xgboost_phase3", "surestimes_oof_base"] == 2384 and choix_fd001.loc["xgboost_phase3", "surestimes_oof_candidat"] == 754
assert candidats_indexes.loc[("lstm", "base"), "surestimes_oof"] == 1503 and candidats_indexes.loc[("lstm", "A"), "surestimes_oof"] == 799
assert choix_fd001.loc["xgboost_phase3", "alertes_manquees_oof_base"] == 552 and choix_fd001.loc["xgboost_phase3", "alertes_manquees_oof_candidat"] == 254
assert candidats_indexes.loc[("lstm", "base"), "alertes_manquees_oof"] == 298 and candidats_indexes.loc[("lstm", "A"), "alertes_manquees_oof"] == 154

# idee 2
assert round(candidats_indexes.loc[("lstm", "B"), "rmse_cv_moyen"], 2) == 11.32
assert round(candidats_indexes.loc[("lstm", "AB"), "rmse_cv_moyen"], 2) == 9.32
assert round(candidats_indexes.loc[("lstm", "AB"), "p_wilcoxon"], 3) == 0.063
assert round(candidats_indexes.loc[("cnn1d", "B"), "gain_rmse"], 2) == 0.89
assert round(candidats_indexes.loc[("cnn1d", "B"), "p_wilcoxon"], 2) == 0.41
assert decisions_reseaux["decisions"]["lstm"]["configuration_retenue"] == "A"
assert decisions_reseaux["decisions"]["cnn1d"]["configuration_retenue"] == "A"

# test officiel FD001 : (brute, plafonnee) de la version retenue
test_attendus = {"LSTM": (12.29, 11.37), "CNN 1D": (12.74, 12.38), "XGBoost": (13.10, 11.88),
                 "LightGBM": (13.40, 12.12), "Random Forest": (14.27, 13.03)}
for nom_modele in test_attendus:
    assert round(tableau_test_sous_jeux.loc[("FD001", nom_modele), "test_brute_retenu"], 2) == test_attendus[nom_modele][0]
    assert round(tableau_test_sous_jeux.loc[("FD001", nom_modele), "test_plafonnee_retenu"], 2) == test_attendus[nom_modele][1]
assert round(tableau_test_sous_jeux.loc[("FD001", "XGBoost"), "test_brute_base"], 2) == 19.70
assert round(tableau_test_sous_jeux.loc[("FD001", "XGBoost"), "test_plafonnee_base"], 2) == 18.53

# FD002 a FD004 : baisse du RMSE test brut des arbres, ecart brut / plafonne
for sous_jeu in ["FD002", "FD003", "FD004"]:
    for nom_modele in ["Random Forest", "XGBoost", "LightGBM"]:
        ligne = tableau_test_sous_jeux.loc[(sous_jeu, nom_modele)]
        assert ligne["test_brute_retenu"] < ligne["test_brute_base"]
assert round(tableau_test_sous_jeux.loc[("FD004", "XGBoost"), "test_brute_base"], 2) == 28.24
assert round(tableau_test_sous_jeux.loc[("FD004", "XGBoost"), "test_brute_retenu"], 2) == 24.76
valeurs_brutes = []
valeurs_plafonnees = []
for sous_jeu in ["FD002", "FD004"]:
    for nom_modele in ["Random Forest", "XGBoost", "LightGBM", "LSTM", "CNN 1D"]:
        valeurs_brutes.append(tableau_test_sous_jeux.loc[(sous_jeu, nom_modele), "test_brute_retenu"])
        valeurs_plafonnees.append(tableau_test_sous_jeux.loc[(sous_jeu, nom_modele), "test_plafonnee_retenu"])
assert round(min(valeurs_brutes), 2) == 22.57 and round(max(valeurs_brutes), 2) == 29.25
assert round(min(valeurs_plafonnees), 2) == 10.95 and round(max(valeurs_plafonnees), 2) == 22.51

# CNN 1D sur FD004
cnn_fd004 = tableau_test_sous_jeux.loc[("FD004", "CNN 1D")]
assert round(cnn_fd004["nasa_test_retenu"]) == 446374
assert cnn_fd004["surestimes_test"] == 55 and cnn_fd004["moteurs_test"] == 248 and cnn_fd004["alertes_manquees_test"] == 23
assert round(tableau_test_sous_jeux.loc[("FD004", "XGBoost"), "nasa_test_retenu"]) == 3721
assert round(tableau_test_sous_jeux.loc[("FD004", "LSTM"), "nasa_test_retenu"]) == 3310
assert round(cnn_fd004["rmse_cv_retenu"], 2) == 11.60
assert round(tableau_test_sous_jeux.loc[("FD004", "LSTM"), "rmse_cv_retenu"], 2) == 11.12
print("Les chiffres cites dans la section F sont bien ceux des fichiers de resultats.")

<a id="partie-g"></a>
## G. Évaluation des versions retenues selon le protocole (FD001)

`docs/protocole_evaluation.md` (règle 11 d'`AGENTS.md`) dit qu'un modèle n'est pas « évalué » sur un seul chiffre. Cette section applique ses paragraphes 2, 3, 4 et 8 aux **6 versions retenues** sur FD001 :

| Paragraphe du protocole | Ce qu'on calcule | Section |
|---|---|---|
| §2 Baselines | 5 modèles volontairement simples, pour savoir ce que vaut un « bon » chiffre | G.2 |
| §3 Régression | 10 métriques, en validation (moyenne ± écart-type des plis) et sur le test (RUL brute et plafonnée) | G.3 |
| §3 Tranches de RUL | l'erreur près de la panne, à mi-vie et loin de la panne | G.4 |
| §4 Classification | « panne dans les N prochains cycles ? » pour N = 15, 30 et 50 | G.5 |
| §8 Comparaisons | tests appariés entre toutes les paires de modèles, avec correction de Bonferroni | G.6 |

**Aucun des 6 modèles n'est réentraîné** : on relit leurs prédictions hors-pli (validation croisée, mêmes plis qu'en A.4) et leurs prédictions test. Seules les 5 baselines sont entraînées ici, sur les mêmes plis (quelques secondes). Ce qui n'est pas fait est listé en G.7.

### G.1 Les prédictions des 6 modèles retenus

**Ce qu'on fait** :
1. on prépare FD001 avec le jeu de 31 variables (`preparer_sous_jeu(..., ecart_initial=20)`) et on vérifie que les lignes, la cible et les plis sont exactement ceux de la section A ;
2. on relit, pour chaque modèle, la version retenue : prédictions hors-pli (une par ligne du train, 20 631) et prédiction au dernier cycle des 100 moteurs test (graine 42 pour les réseaux, le modèle publié) ;
3. on vérifie qu'elles sont identiques aux fichiers du contrat (`sorties/predictions/`) et qu'elles redonnent les RMSE publiés.

**Comment lire le résultat** : une ligne par modèle, RMSE de validation (moyenne des 5 plis) et RMSE test contre la RUL brute.

In [ ]:
# 1) FD001 avec les 31 variables de Matisse ; memes lignes, meme ordre que la section A
contexte_31 = preparer_sous_jeu("FD001", fenetre=5, plafond=PLAFOND_RUL, ecart_initial=20)
X_31 = contexte_31["X"]                              # 31 variables, une ligne par cycle
y_31 = contexte_31["y"].to_numpy(dtype=float)        # RUL plafonnee a 125
groupes_31 = contexte_31["groups"].to_numpy()        # numero de moteur de chaque ligne
assert X_31.shape == (20631, 31)
assert np.array_equal(groupes_31, groups.to_numpy())       # memes moteurs, meme ordre qu'en A.3
assert np.allclose(y_31, y.to_numpy(dtype=float))          # meme cible
plis_31 = calculer_plis(X_31, y_31, groupes_31, n_splits=5, random_state=42)
for numero_pli in range(5):  # les plis de A.4 sont bien ceux des scripts : on les reutilise tels quels
    assert np.array_equal(plis_31[numero_pli][1], plis[numero_pli][1])

cles_lignes = contexte_31["train"][["unit_number", "time_cycles"]].reset_index(drop=True)  # cle de chaque ligne
y_test_brute = np.asarray(contexte_31["y_test_vraie"], dtype=float)  # RUL vraie de RUL_FD001.txt
assert np.array_equal(y_test_brute, y_test_vraie)                   # memes 100 moteurs test qu'en A.3
y_test_plafonnee = np.minimum(y_test_brute, PLAFOND_RUL)             # la meme, plafonnee a 125
moteurs_test_31 = np.asarray(contexte_31["moteurs_test"])            # ordre des moteurs test

# 2) predictions des 4 modeles du script des arbres (version retenue par la regle)
oof = {}   # cle -> predictions hors-pli, une par ligne du train
test = {}  # cle -> prediction au dernier cycle de chaque moteur test
oof_arbres = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "05_arbres_oof.csv.gz"))
oof_arbres = oof_arbres[oof_arbres["sous_jeu"] == "FD001"]
test_arbres = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "05_arbres_test_dernier_cycle.csv"))
test_arbres = test_arbres[test_arbres["sous_jeu"] == "FD001"]
for cle in CLES_ARBRES:
    jeu = choix_fd001.loc[cle, "jeu_retenu"]  # "base" ou "ecart_initial_20"
    selection = oof_arbres[(oof_arbres["modele"] == cle) & (oof_arbres["jeu_features"] == jeu)]
    # fusion sur (moteur, cycle) : les predictions sont remises dans l'ordre exact des lignes du train
    fusion = cles_lignes.merge(selection, on=["unit_number", "time_cycles"], how="left", validate="one_to_one")
    assert fusion["rul_predite_oof"].notna().all()           # une prediction pour chaque ligne
    assert np.allclose(fusion["rul_cappee"].to_numpy(), y_31)  # meme cible que la section A
    oof[cle] = fusion["rul_predite_oof"].to_numpy(dtype=float)
    selection_test = test_arbres[(test_arbres["modele"] == cle) & (test_arbres["jeu_features"] == jeu)]
    test[cle] = selection_test.set_index("unit_number").loc[moteurs_test_31, "rul_predite"].to_numpy(dtype=float)

# predictions des 2 reseaux (version A, graine 42 pour le test)
oof_reseaux = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_oof_fd001.csv.gz"))
fusion_reseaux = cles_lignes.merge(oof_reseaux, on=["unit_number", "time_cycles"], how="left", validate="one_to_one")
assert np.allclose(fusion_reseaux["RUL_cappee"].to_numpy(), y_31)
test_reseaux_fd001 = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_predictions_test_fd001.csv"))
test_reseaux_fd001 = test_reseaux_fd001.set_index("unit_number").loc[moteurs_test_31]
for cle in CLES_RESEAUX:
    # la colonne oof_<reseau> du fichier est bien la version retenue (candidat A)
    assert np.array_equal(fusion_reseaux[f"oof_{cle}"].to_numpy(), fusion_reseaux[f"oof_{cle}_A"].to_numpy())
    oof[cle] = fusion_reseaux[f"oof_{cle}"].to_numpy(dtype=float)
    test[cle] = test_reseaux_fd001[f"{cle}_graine42"].to_numpy(dtype=float)

# 3) controle croise avec les fichiers du contrat et avec les metriques publiees
for cle in MODELES_RETENUS:
    predictions_contrat = lire_predictions("FD001", cle)[0]  # toutes les predictions test, chaque cycle
    dernier_cycle = extraire_dernier_cycle(predictions_contrat).set_index("unit_number").loc[moteurs_test_31]
    assert np.allclose(dernier_cycle["rul_predite"].to_numpy(), test[cle], atol=1e-3)
    assert np.allclose(dernier_cycle["rul_vraie"].to_numpy(), y_test_brute)
    metadonnees = lire_metadonnees("FD001", cle)
    rmse_test_recalcule = rmse(y_test_brute, test[cle])
    assert abs(rmse_test_recalcule - metadonnees["metriques_test"]["rmse"]) < 1e-3
    rmse_cv_recalcule = metriques_regression_par_pli(y_31, oof[cle], plis)["rmse"].mean()
    print(f"{MODELES_RETENUS[cle]:16s} : RMSE CV (5 plis) = {rmse_cv_recalcule:6.3f} ; RMSE test (RUL brute) = "
          f"{rmse_test_recalcule:6.3f}")

### G.2 Les baselines : des modèles volontairement simples

**Pourquoi** : un RMSE de 9 cycles n'a de sens que comparé à ce que ferait un modèle sans intelligence. Le protocole (§2) en demande plusieurs :

| Baseline | Ce qu'elle prédit |
|---|---|
| Moyenne | toujours la RUL moyenne du train (c'est exactement la « référence naïve » des sections B à F) |
| Médiane | toujours la RUL médiane du train |
| Constante 125 | toujours 125 : « le moteur est neuf » |
| Aléatoire | un nombre tiré au hasard entre 0 et 125 (graine fixée) |
| Ridge | une **régression linéaire** sur les 31 variables (avec une petite pénalité, `alpha = 1`, non réglée) ; les variables sont centrées-réduites, réglage appris dans chaque pli |

**Ce qu'on fait** : chaque baseline est entraînée sur les 80 moteurs d'entraînement de chaque pli, puis prédit les 20 moteurs de validation ; pour le test, elle est entraînée sur les 100 moteurs. **Comment lire** : erreur sur les moteurs d'entraînement (moyenne des 5 plis) contre erreur sur les moteurs de validation ; un écart faible veut dire que la baseline n'apprend pas par cœur, un niveau élevé qu'elle est trop simple (sous-apprentissage).

In [ ]:
BASELINES = ["Moyenne", "Mediane", "Constante 125", "Aleatoire uniforme", "Ridge"]
GRAINE_BASELINES = 42  # graine du tirage de la baseline aleatoire


def fabrique_baseline(nom):
    """
    Renvoie une baseline neuve (non entrainee), sauf l'aleatoire qui est traitee a part.

    Parameters
    ----------
    nom : str
        "Moyenne", "Mediane", "Constante 125" ou "Ridge".

    Returns
    -------
    estimateur scikit-learn
        Le modele non entraine ; Ridge est precede d'un StandardScaler appris sur le train du pli.
    """
    if nom == "Moyenne":
        return DummyRegressor(strategy="mean")
    if nom == "Mediane":
        return DummyRegressor(strategy="median")
    if nom == "Constante 125":
        return DummyRegressor(strategy="constant", constant=PLAFOND_RUL)
    if nom == "Ridge":
        # centrer-reduire chaque variable (time_cycles n'est pas entre 0 et 1), puis regression lineaire, alpha non regle
        return Pipeline([("echelle", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
    raise ValueError(f"baseline inconnue : {nom}")


metriques_train_baselines = {}  # nom -> moyenne et ecart-type des metriques d'entrainement des 5 plis
for nom in BASELINES:
    predictions_oof = np.full(len(y_31), np.nan)  # sera rempli pli par pli
    lignes_train = []  # metriques sur les moteurs d'entrainement, une ligne par pli
    generateur_val = np.random.default_rng(GRAINE_BASELINES)  # aleatoire : graine fixe, un tirage par pli
    generateur_train = np.random.default_rng(GRAINE_BASELINES + 1)
    for idx_train, idx_val in plis:
        if nom == "Aleatoire uniforme":
            predictions_oof[idx_val] = generateur_val.uniform(0, PLAFOND_RUL, size=len(idx_val))
            predictions_train = generateur_train.uniform(0, PLAFOND_RUL, size=len(idx_train))
        else:
            modele = fabrique_baseline(nom)
            modele.fit(X_31.iloc[idx_train], y_31[idx_train])  # appris sur les moteurs d'entrainement du pli
            predictions_oof[idx_val] = modele.predict(X_31.iloc[idx_val])
            predictions_train = modele.predict(X_31.iloc[idx_train])
        lignes_train.append(metriques_regression(y_31[idx_train], predictions_train))
    assert not np.isnan(predictions_oof).any()  # chaque ligne a recu sa prediction hors-pli
    oof[nom] = predictions_oof
    metriques_train_baselines[nom] = resumer_plis(pd.DataFrame(lignes_train), colonne_pli="aucune")

    # test : modele appris sur les 100 moteurs, prediction au dernier cycle, ramenee a 0 si negative (comme les modeles)
    if nom == "Aleatoire uniforme":
        predictions_test = np.random.default_rng(GRAINE_BASELINES).uniform(0, PLAFOND_RUL, size=len(y_test_brute))
    else:
        modele = fabrique_baseline(nom)
        modele.fit(X_31, y_31)
        predictions_test = modele.predict(contexte_31["X_test_dernier"])
    test[nom] = np.clip(np.asarray(predictions_test, dtype=float), a_min=0, a_max=None)

# la reference naive des sections B a F EST la baseline "Moyenne" : verification
assert np.allclose(oof["Moyenne"], oof["reference_naive"])
assert np.allclose(test["Moyenne"], test["reference_naive"])

lignes = []
for nom in BASELINES:
    validation = resumer_plis(metriques_regression_par_pli(y_31, oof[nom], plis))  # memes metriques sur la validation
    ligne = {}
    ligne["baseline"] = nom
    ligne["rmse_train"] = metriques_train_baselines[nom]["rmse_moyenne"]
    ligne["rmse_validation"] = validation["rmse_moyenne"]
    ligne["mae_train"] = metriques_train_baselines[nom]["mae_moyenne"]
    ligne["mae_validation"] = validation["mae_moyenne"]
    ligne["r2_train"] = metriques_train_baselines[nom]["r2_moyenne"]
    ligne["r2_validation"] = validation["r2_moyenne"]
    lignes.append(ligne)
tableau_baselines = pd.DataFrame(lignes).set_index("baseline")
tableau_baselines.round(3)

### G.3 Métriques de régression

**Les métriques**, pour un écart e = prédiction − vérité (en cycles) :

| Métrique | Sens | Exemple |
|---|---|---|
| MAE | moyenne des \|e\| : l'erreur typique | erreurs de 5 et 15 cycles → MAE = 10 |
| RMSE | racine de la moyenne des e² : pénalise davantage les grosses erreurs | 5 et 15 → RMSE = 11,2 |
| R² | part de la variation de la RUL expliquée : 1 = parfait, 0 = pas mieux que la moyenne, négatif = pire | |
| erreur abs. médiane | la moitié des erreurs sont plus petites : insensible à quelques erreurs énormes | |
| erreur max | la pire erreur | |
| biais | moyenne des e : positif = le modèle surestime en moyenne (dangereux), négatif = il sous-estime (prudent) | |
| NASA somme / moyen | score asymétrique 13/10 (section C) : somme des pénalités, et pénalité moyenne par prédiction | |
| surestimés > 20 | nombre et % de prédictions qui surestiment de plus de 20 cycles | |

**Comment lire** : validation = moyenne ± écart-type des 5 plis, chaque pli comptant environ 4 100 lignes (tous les cycles des 20 moteurs de validation). Les prédictions hors-pli des arbres et de Ridge ne sont pas ramenées à 0, celles des réseaux le sont (choix des scripts) ; sur le test, tout est ramené à 0.

In [ ]:
NOMS_AFFICHES = {}  # cle -> nom affiche, pour les 6 modeles et les 5 baselines
for cle in MODELES_RETENUS:
    NOMS_AFFICHES[cle] = MODELES_RETENUS[cle]
for nom in BASELINES:
    NOMS_AFFICHES[nom] = nom
ORDRE_MODELES = list(MODELES_RETENUS) + BASELINES  # les 6 modeles retenus, puis les 5 baselines

resume_validation = {}         # cle -> moyenne et ecart-type des metriques des 5 plis
metriques_test_brute = {}      # cle -> metriques au dernier cycle, contre la RUL brute
metriques_test_plafonnee = {}  # cle -> les memes, contre la RUL plafonnee a 125
for cle in ORDRE_MODELES:
    resume_validation[cle] = resumer_plis(metriques_regression_par_pli(y_31, oof[cle], plis))
    metriques_test_brute[cle] = metriques_regression(y_test_brute, test[cle])
    metriques_test_plafonnee[cle] = metriques_regression(y_test_plafonnee, test[cle])

# (cle de la metrique, nom de colonne, decimales affichees)
COLONNES_REGRESSION = [("mae", "MAE", 2), ("rmse", "RMSE", 2), ("r2", "R2", 3),
                       ("erreur_absolue_mediane", "erreur abs. mediane", 2), ("erreur_max", "erreur max", 1),
                       ("biais", "biais", 2), ("nasa_somme", "NASA somme", 0), ("nasa_moyen", "NASA moyen", 2),
                       ("surestimes", "surestimes > 20", 1), ("surestimes_pct", "% surestimes", 2)]

lignes = []
for cle in ORDRE_MODELES:
    ligne = {}
    ligne["modele"] = NOMS_AFFICHES[cle]
    for metrique, colonne, decimales in COLONNES_REGRESSION:
        ligne[colonne] = formater_moyenne_ecart(resume_validation[cle][metrique + "_moyenne"],
                                                resume_validation[cle][metrique + "_ecart_type"], decimales)
    lignes.append(ligne)
tableau_regression_validation = pd.DataFrame(lignes).set_index("modele")
print("Validation croisee (moyenne ± ecart-type des 5 plis) :")
tableau_regression_validation

**Sur le test officiel** (100 moteurs, une prédiction par moteur au dernier cycle observé, évaluée une fois) : d'abord contre la RUL **brute** de `RUL_FD001.txt`, puis contre la RUL **plafonnée à 125**. Seuls 11 moteurs test ont une vraie RUL au-delà de 125 : les deux tableaux diffèrent peu sur FD001.

In [ ]:
lignes_brute = []
lignes_plafonnee = []
for cle in ORDRE_MODELES:
    ligne_brute = {}
    ligne_brute["modele"] = NOMS_AFFICHES[cle]
    ligne_plafonnee = {}
    ligne_plafonnee["modele"] = NOMS_AFFICHES[cle]
    for metrique, colonne, decimales in COLONNES_REGRESSION:
        decimales_test = decimales
        if metrique == "nasa_somme":
            decimales_test = 1  # 100 moteurs seulement : une decimale reste lisible
        ligne_brute[colonne] = round(metriques_test_brute[cle][metrique], decimales_test)
        ligne_plafonnee[colonne] = round(metriques_test_plafonnee[cle][metrique], decimales_test)
    lignes_brute.append(ligne_brute)
    lignes_plafonnee.append(ligne_plafonnee)
tableau_test_brute = pd.DataFrame(lignes_brute).set_index("modele")
tableau_test_plafonnee = pd.DataFrame(lignes_plafonnee).set_index("modele")
print(f"Moteurs test dont la vraie RUL depasse 125 : {int(np.sum(y_test_brute > PLAFOND_RUL))} sur {len(y_test_brute)}")
print("\nTest officiel, contre la RUL brute :")
display(tableau_test_brute)
print("Test officiel, contre la RUL plafonnee a 125 :")
display(tableau_test_plafonnee)

**La MAPE (erreur relative moyenne, en %) : calculée, puis écartée.** Elle divise chaque erreur par la vraie RUL. Or chaque moteur de validation finit à RUL = 0 (son dernier cycle) : on divise par 0 (scikit-learn remplace 0 par un nombre minuscule), d'où des valeurs absurdes. Sur le test, la plus petite vraie RUL n'est pas nulle : la MAPE est finie, mais dominée par les moteurs proches de la panne. Elle n'est pas comparable entre validation et test et ne mesure pas l'erreur utile : on ne la garde pas.

In [ ]:
print(f"Lignes de RUL vraie nulle par pli de validation : {resume_validation['xgboost_phase3']['nb_rul_vraie_nulle_moyenne']:.0f}")
print(f"Plus petite vraie RUL du test : {y_test_brute.min():.0f} cycles\n")
for cle in ORDRE_MODELES:
    print(f"{NOMS_AFFICHES[cle]:18s} : MAPE validation = {resume_validation[cle]['mape_moyenne']:.1e} ; "
          f"MAPE test = {metriques_test_brute[cle]['mape']:.3f}")

### G.4 L'erreur selon la distance à la panne (tranches de RUL vraie)

**Pourquoi** : une erreur de 10 cycles n'a pas la même gravité à 20 cycles de la panne qu'à 100. Le protocole demande l'erreur par tranche de vraie RUL : ≤ 30 (zone d'alerte), 31 à 60, 61 à 125, et > 125 (seulement sur le test : la cible d'entraînement ne dépasse pas 125).

**Comment lire** : chaque case donne « MAE / RMSE / biais » en cycles ; un biais positif dans la tranche ≤ 30 serait le cas le plus dangereux (le modèle croit le moteur plus jeune qu'il n'est). La ligne `n` donne le nombre de lignes (validation) ou de moteurs (test) de chaque tranche.

In [ ]:
lignes_tranches = []
for cle in ORDRE_MODELES:
    for ensemble, vrais, predits in [("validation", y_31, oof[cle]), ("test (RUL brute)", y_test_brute, test[cle])]:
        tranches = erreurs_par_tranche_rul(vrais, predits)  # une ligne par tranche
        for indice in tranches.index:
            source = tranches.loc[indice]
            if source["n"] == 0:
                continue  # tranche vide (> 125 en validation : la cible est plafonnee)
            ligne = {}
            ligne["modele"] = NOMS_AFFICHES[cle]
            ligne["ensemble"] = ensemble
            ligne["tranche"] = source["tranche"]
            ligne["n"] = int(source["n"])
            ligne["mae"] = source["mae"]
            ligne["rmse"] = source["rmse"]
            ligne["biais"] = source["biais"]
            ligne["mae / rmse / biais"] = f"{source['mae']:.2f} / {source['rmse']:.2f} / {source['biais']:+.2f}"
            lignes_tranches.append(ligne)
tableau_tranches = pd.DataFrame(lignes_tranches)

noms_dans_l_ordre = []  # ordre d'affichage des lignes : celui de ORDRE_MODELES
for cle in ORDRE_MODELES:
    noms_dans_l_ordre.append(NOMS_AFFICHES[cle])
for ensemble in ["validation", "test (RUL brute)"]:
    sous_tableau = tableau_tranches[tableau_tranches["ensemble"] == ensemble]
    ordre_tranches = list(pd.unique(sous_tableau["tranche"]))  # tranches dans l'ordre croissant de RUL
    pivot = sous_tableau.pivot(index="modele", columns="tranche", values="mae / rmse / biais")
    pivot = pivot.loc[noms_dans_l_ordre, ordre_tranches]
    effectifs_xgb = sous_tableau[sous_tableau["modele"] == "XGBoost"].set_index("tranche")
    effectifs = {}  # tranche -> nombre de lignes (validation) ou de moteurs (test)
    for tranche in ordre_tranches:
        effectifs[tranche] = int(effectifs_xgb.loc[tranche, "n"])
    print(f"{ensemble} — MAE / RMSE / biais par tranche de vraie RUL ; n = {effectifs}")
    display(pivot)

### G.5 Classification : « ce moteur tombe-t-il en panne dans les N prochains cycles ? »

**L'idée** : en maintenance, la vraie question est souvent oui / non. On transforme chaque prédiction en **alerte** : alerte si la RUL prédite ≤ un seuil. La vérité est « positif » si la vraie RUL ≤ N. On étudie N = 15, 30 et 50 cycles.

**Les mesures** :
- **matrice de confusion** `vn/fp/fn/vp` : vrais négatifs (pas d'alerte, à raison), faux positifs (alerte inutile : coûteux), faux négatifs (alerte manquée : **dangereux**), vrais positifs ;
- **précision** = vp / (vp + fp) : parmi les alertes, la part de justes ; **rappel** = vp / (vp + fn) : parmi les moteurs à risque, la part détectée ; **F1** = moyenne harmonique des deux ;
- **AUC ROC** et **précision moyenne (AP)** : qualité du classement des moteurs du plus au moins à risque, tous seuils confondus (1 = parfait, AUC 0,5 = hasard) ;
- l'**accuracy** (part de bonnes réponses) est calculée mais trompe ici : les positifs sont rares, un modèle qui n'alerte jamais a déjà une accuracy élevée.

**Deux seuils** : le seuil « défaut » = N ; le seuil « choisi » est fixé sur les prédictions hors-pli de la validation (parmi les seuils qui donnent un rappel ≥ 0,95, celui qui a la meilleure précision ; à précision égale, le plus grand rappel), puis appliqué tel quel au test, qui ne sert jamais à le choisir. `R plis` = rappel moyen ± écart-type entre les 5 plis.

**Comment lire** : un tableau par N ; les trois baselines constantes (Moyenne = référence naïve, Médiane, Constante 125) sont résumées sous les tableaux.

In [ ]:
HORIZONS = [15, 30, 50]  # N : panne dans les N prochains cycles
RAPPEL_MINIMUM = 0.95    # rappel exige pour le seuil choisi sur la validation

lignes_classification = []
for cle in ORDRE_MODELES:
    for horizon in HORIZONS:
        choix_seuil = choisir_seuil_alerte(y_31, oof[cle], horizon, rappel_min=RAPPEL_MINIMUM)  # sur la validation
        for regle_seuil, seuil in [("defaut (= N)", float(horizon)), ("choisi sur validation", choix_seuil["seuil_alerte"])]:
            validation = metriques_classification_alerte(y_31, oof[cle], horizon, seuil)  # hors-pli, 5 plis reunis
            rappels_plis = []  # rappel de chaque pli, pour mesurer sa variation
            for idx_train, idx_val in plis:
                rappels_plis.append(metriques_classification_alerte(y_31[idx_val], oof[cle][idx_val], horizon, seuil)["rappel"])
            resultat_test = metriques_classification_alerte(y_test_brute, test[cle], horizon, seuil)
            ligne = {}
            ligne["modele"] = NOMS_AFFICHES[cle]
            ligne["N"] = horizon
            ligne["regle_seuil"] = regle_seuil
            ligne["seuil"] = seuil
            for nom_mesure in ["vn", "fp", "fn", "vp", "nb_positifs", "proportion_positifs", "accuracy", "precision",
                               "rappel", "f1", "auc_roc", "precision_moyenne"]:
                ligne["val_" + nom_mesure] = validation[nom_mesure]
                ligne["test_" + nom_mesure] = resultat_test[nom_mesure]
            ligne["val_rappel_plis_moyenne"] = float(np.mean(rappels_plis))
            ligne["val_rappel_plis_ecart_type"] = float(np.std(rappels_plis))  # ddof = 0, comme resumer_plis
            lignes_classification.append(ligne)
tableau_classification = pd.DataFrame(lignes_classification)

CONSTANTES = ["Reference naive", "Moyenne", "Mediane", "Constante 125"]  # baselines qui predisent une constante
for horizon in HORIZONS:
    sous_tableau = tableau_classification[(tableau_classification["N"] == horizon)
                                          & (~tableau_classification["modele"].isin(CONSTANTES))]
    lignes = []
    for indice in sous_tableau.index:
        source = sous_tableau.loc[indice]
        ligne = {}
        ligne["modele"] = source["modele"]
        ligne["seuil"] = f"{source['seuil']:.2f} ({source['regle_seuil']})"
        ligne["val vn/fp/fn/vp"] = f"{source['val_vn']}/{source['val_fp']}/{source['val_fn']}/{source['val_vp']}"
        ligne["val P"] = round(source["val_precision"], 3)
        ligne["val R"] = round(source["val_rappel"], 3)
        ligne["val F1"] = round(source["val_f1"], 3)
        ligne["R plis"] = formater_moyenne_ecart(source["val_rappel_plis_moyenne"], source["val_rappel_plis_ecart_type"], 3)
        ligne["val AUC"] = round(source["val_auc_roc"], 4)
        ligne["val AP"] = round(source["val_precision_moyenne"], 3)
        ligne["test vn/fp/fn/vp"] = f"{source['test_vn']}/{source['test_fp']}/{source['test_fn']}/{source['test_vp']}"
        ligne["test P"] = round(source["test_precision"], 3)
        ligne["test R"] = round(source["test_rappel"], 3)
        ligne["test F1"] = round(source["test_f1"], 3)
        ligne["test AUC"] = round(source["test_auc_roc"], 3)
        ligne["test AP"] = round(source["test_precision_moyenne"], 3)
        lignes.append(ligne)
    premiere = sous_tableau.iloc[0]
    print(f"N = {horizon} : positifs = {premiere['val_nb_positifs']} lignes de validation "
          f"({100 * premiere['val_proportion_positifs']:.1f} %), {premiere['test_nb_positifs']} moteurs test sur 100")
    display(pd.DataFrame(lignes).set_index("modele"))

# les baselines constantes : jamais d'alerte au seuil N, alerte partout au seuil choisi
for horizon in HORIZONS:
    naive_defaut = tableau_classification[(tableau_classification["modele"] == "Reference naive")
                                          & (tableau_classification["N"] == horizon)
                                          & (tableau_classification["regle_seuil"] == "defaut (= N)")].iloc[0]
    print(f"N = {horizon} : la reference naive n'alerte jamais au seuil N -> rappel {naive_defaut['val_rappel']:.0f}, "
          f"accuracy de validation {naive_defaut['val_accuracy']:.3f} (= part des negatifs)")

### G.6 Les écarts entre modèles sont-ils significatifs ? Toutes les paires

**Ce qu'on fait** : on compare les 6 modèles retenus et Ridge **deux à deux** (21 paires), moteur par moteur (test apparié, comme en B.2). **Règle écrite avant les résultats** (constante `REGLE_COMPARAISON` ci-dessous) :

> A est déclaré meilleur que B si la p-valeur de Wilcoxon bilatérale, **multipliée par 21** (correction de Bonferroni), est < 0,05 et que la différence moyenne A − B est négative. Sinon : pas de différence significative. Les p-valeurs du test t sont rapportées à côté.

Trois jeux de valeurs appariées : en validation, le RMSE hors-pli de chacun des 100 moteurs d'entraînement ; sur le test, l'erreur absolue de chacun des 100 moteurs test ; pour information, le RMSE des 5 plis (avec 5 paires seulement, la p-valeur de Wilcoxon bilatérale ne peut pas descendre sous 0,0625 : aucun écart ne peut y être significatif).

**Comment lire** : `difference_moyenne` = A − B (négatif = A se trompe moins) ; `A_meilleur_sur` = nombre de moteurs où A fait mieux ; `p_W_corrigee` = p-valeur de Wilcoxon multipliée par 21 (plafonnée à 1).

In [ ]:
REGLE_COMPARAISON = (
    "Comparaisons appariees par moteur entre les 6 modeles retenus et Ridge (21 paires) : test t apparie et "
    "Wilcoxon signe bilateral (comparer_apparie), p-valeurs multipliees par le nombre de paires (Bonferroni). "
    "A est declare meilleur que B si p Wilcoxon corrigee < 0.05 et difference moyenne A - B < 0 ; sinon pas de "
    "difference significative."
)
SEUIL_P = 0.05

modeles_compares = ["reference_naive", "random_forest", "lightgbm", "xgboost_phase3", "lstm", "cnn1d", "Ridge"]
valeurs_validation = {}  # nom -> RMSE hors-pli de chaque moteur d'entrainement (100)
valeurs_test = {}        # nom -> erreur absolue de chaque moteur test (100)
valeurs_plis = {}        # nom -> RMSE de chacun des 5 plis
for cle in modeles_compares:
    valeurs_validation[NOMS_AFFICHES[cle]] = erreurs_par_moteur(y_31, oof[cle], groupes_31)["rmse"].to_numpy()
    valeurs_test[NOMS_AFFICHES[cle]] = np.abs(test[cle] - y_test_brute)
    valeurs_plis[NOMS_AFFICHES[cle]] = metriques_regression_par_pli(y_31, oof[cle], plis)["rmse"].to_numpy()

comparaisons = {}  # jeu de valeurs -> tableau des 21 paires
comparaisons["validation"] = comparer_modeles_par_paires(valeurs_validation, seuil_p=SEUIL_P)
comparaisons["test"] = comparer_modeles_par_paires(valeurs_test, seuil_p=SEUIL_P)
comparaisons["plis"] = comparer_modeles_par_paires(valeurs_plis, seuil_p=SEUIL_P)


def mettre_en_forme_comparaisons(tableau):
    """
    Met en forme un tableau de comparaisons par paires pour l'affichage.

    Parameters
    ----------
    tableau : pandas.DataFrame
        Le resultat de `comparer_modeles_par_paires`.

    Returns
    -------
    pandas.DataFrame
        Une ligne par paire (index "A vs B") : difference moyenne, nombre d'unites ou A est meilleur,
        p-valeurs corrigees (texte) et conclusion.
    """
    lignes = []
    for indice in tableau.index:
        source = tableau.loc[indice]
        ligne = {}
        ligne["paire"] = f"{source['modele_a']} vs {source['modele_b']}"
        ligne["difference_moyenne"] = round(source["difference_moyenne"], 2)
        ligne["A_meilleur_sur"] = f"{source['nb_unites_a_meilleur']}/{source['n']}"
        ligne["p_t_corrigee"] = f"{source['p_t_bonferroni']:.2g}"
        ligne["p_W_corrigee"] = f"{source['p_wilcoxon_bonferroni']:.2g}"
        ligne["conclusion"] = source["conclusion"]
        lignes.append(ligne)
    return pd.DataFrame(lignes).set_index("paire")


print("Validation : RMSE hors-pli par moteur (100 moteurs d'entrainement)")
display(mettre_en_forme_comparaisons(comparaisons["validation"]))
print("Test : erreur absolue par moteur (100 moteurs test)")
display(mettre_en_forme_comparaisons(comparaisons["test"]))
nb_significatives_plis = int(np.sum(comparaisons["plis"]["conclusion"] != "pas de difference significative"))
print(f"Pour information, RMSE des 5 plis : {nb_significatives_plis} paire(s) significative(s) sur 21 ; "
      f"plus petite p-valeur de Wilcoxon brute = {comparaisons['plis']['p_wilcoxon'].min():.4f}")

### G.7 Ce qu'il faut retenir de la section G

- **Les 5 modèles qui apprennent battent nettement les baselines** (la 6e version retenue, la référence naïve, est la baseline « Moyenne »). RMSE de validation de 8,94 (LSTM) à 10,19 (Random Forest), contre 16,95 pour Ridge, la meilleure baseline, et 41,69 pour la moyenne. Toutes les comparaisons contre la référence naïve et contre Ridge sont significatives, en validation comme sur le test. Ridge, simple régression linéaire, explique déjà une grande part de la variation (R² de validation 0,834) : les modèles non linéaires divisent son erreur par 1,7 à 1,9 environ.
- **Les baselines ne sur-apprennent pas** (Ridge : RMSE 15,70 sur les moteurs d'entraînement, 16,95 en validation). Pour les 6 modèles retenus, l'erreur sur les moteurs d'entraînement n'est pas calculée ici (voir la liste ci-dessous).
- **Près de la panne, l'erreur est petite.** Tranche de vraie RUL ≤ 30 : RMSE de validation de 2,95 (LSTM) à 4,55 (CNN 1D), et sur le test de 1,62 (LSTM) à 4,06 (XGBoost). L'erreur vient surtout de la tranche 61-125 et des 11 moteurs test de vraie RUL > 125, que tous les modèles sous-estiment (biais de −8,51 pour le CNN 1D à −24,24 pour Random Forest) : c'est l'effet du plafond d'entraînement.
- **Classification à N = 30** : au seuil N, les modèles retenus détectent en validation de 90 % (Random Forest) à 95 % (LSTM) des lignes à risque, avec une précision d'environ 0,96 ; AUC ≥ 0,997. Sur le test, avec le seuil choisi sur la validation, le rappel va de 0,92 à 1,00. Une référence qui n'alerte jamais a une accuracy de 0,850 : l'accuracy est écartée.
- **Comparaisons par paires** : en validation, le LSTM bat Random Forest et le CNN 1D, et XGBoost et LightGBM battent Random Forest ; LSTM contre XGBoost ou LightGBM : non significatif après correction (p de Wilcoxon corrigée 0,71 et 0,52). **Sur le test, aucune des 10 paires entre les 5 modèles appris n'est significative** : 100 moteurs ne suffisent pas à les départager.

**Ce que le protocole demande et qui n'est pas fait ici** (règle 11 : à signaler) :
- les métriques sur les moteurs d'entraînement des 6 modèles (§3, §5.1) : il faudrait réentraîner un modèle par pli (environ 1 minute pour les arbres, 20 minutes pour les réseaux) ;
- un classifieur dédié (§4, option 2 : régression logistique ou XGBClassifier validés par `StratifiedGroupKFold`) : seule la lecture des régressions comme alertes est faite ;
- les courbes d'apprentissage et de validation (§5), l'optimisation des hyperparamètres des 6 modèles et de l'`alpha` de Ridge (§6 ; ici `alpha = 1`), la validation croisée répétée sur plusieurs graines (§8) : chacun demande de nombreux réentraînements ;
- les figures ROC et précision-rappel (§9) et ce protocole sur FD002 à FD004 ;
- **règle 1.3 du protocole** : la mise à l'échelle min-max (et, pour les réseaux, celle de `ecartinit_20`) est apprise sur tout le train **avant** la validation croisée, pas dans chaque pli (aucune information de la cible n'y entre, mais les moteurs de validation contribuent aux bornes).

**À savoir pour lire les chiffres** : les prédictions hors-pli des arbres et de Ridge ne sont pas ramenées à 0, celles des réseaux le sont ; sur le test, les réseaux sont évalués avec la seule graine 42 (le modèle publié) ; à N = 15, le test ne compte que 10 moteurs positifs, un seul moteur fait donc varier le rappel de 0,1.

**Contrôle des chiffres de la section G** : comme en F, la cellule suivante vérifie les chiffres cités ci-dessus.

In [ ]:
rmse_validation_retenus = []
for cle in ["random_forest", "lightgbm", "xgboost_phase3", "lstm", "cnn1d"]:
    rmse_validation_retenus.append(resume_validation[cle]["rmse_moyenne"])
assert round(min(rmse_validation_retenus), 2) == 8.94 and round(resume_validation["lstm"]["rmse_moyenne"], 2) == 8.94
assert round(max(rmse_validation_retenus), 2) == 10.19 and round(resume_validation["random_forest"]["rmse_moyenne"], 2) == 10.19
assert round(resume_validation["Ridge"]["rmse_moyenne"], 2) == 16.95
assert round(resume_validation["reference_naive"]["rmse_moyenne"], 2) == 41.69
for nom in ["Moyenne", "Mediane", "Constante 125", "Aleatoire uniforme"]:  # Ridge est bien la meilleure baseline
    assert resume_validation[nom]["rmse_moyenne"] > resume_validation["Ridge"]["rmse_moyenne"]
assert round(resume_validation["Ridge"]["r2_moyenne"], 3) == 0.834
rapports = []  # RMSE de Ridge / RMSE de chaque modele non lineaire
for valeur in rmse_validation_retenus:
    rapports.append(resume_validation["Ridge"]["rmse_moyenne"] / valeur)
assert round(min(rapports), 1) == 1.7 and round(max(rapports), 1) == 1.9
assert round(metriques_train_baselines["Ridge"]["rmse_moyenne"], 2) == 15.70

# chaque modele appris bat chaque baseline en RMSE, MAE, R2 et score NASA (validation, test brut et plafonne)
for cle in ["random_forest", "lightgbm", "xgboost_phase3", "lstm", "cnn1d"]:
    for nom in BASELINES:
        for metrique in ["rmse", "mae", "nasa_somme"]:  # plus petit = mieux
            assert resume_validation[cle][metrique + "_moyenne"] < resume_validation[nom][metrique + "_moyenne"]
            assert metriques_test_brute[cle][metrique] < metriques_test_brute[nom][metrique]
            assert metriques_test_plafonnee[cle][metrique] < metriques_test_plafonnee[nom][metrique]
        assert resume_validation[cle]["r2_moyenne"] > resume_validation[nom]["r2_moyenne"]  # plus grand = mieux
        assert metriques_test_brute[cle]["r2"] > metriques_test_brute[nom]["r2"]
        assert metriques_test_plafonnee[cle]["r2"] > metriques_test_plafonnee[nom]["r2"]

# toutes les comparaisons contre la reference naive et Ridge sont significatives (validation et test)
for jeu in ["validation", "test"]:
    tableau = comparaisons[jeu]
    for indice in tableau.index:
        paire = tableau.loc[indice]
        if paire["modele_a"] in ["Reference naive", "Ridge"] or paire["modele_b"] in ["Reference naive", "Ridge"]:
            assert paire["conclusion"] != "pas de difference significative", (jeu, paire["modele_a"], paire["modele_b"])

# tranches de RUL
tranche_30 = tableau_tranches[tableau_tranches["tranche"] == "<= 30"]
appris = ["Random Forest", "LightGBM", "XGBoost", "LSTM", "CNN 1D"]
for ensemble, minimum, modele_min, maximum, modele_max in [("validation", 2.95, "LSTM", 4.55, "CNN 1D"),
                                                         ("test (RUL brute)", 1.62, "LSTM", 4.06, "XGBoost")]:
    sous = tranche_30[(tranche_30["ensemble"] == ensemble) & (tranche_30["modele"].isin(appris))].set_index("modele")
    assert round(sous["rmse"].min(), 2) == minimum and sous["rmse"].idxmin() == modele_min
    assert round(sous["rmse"].max(), 2) == maximum and sous["rmse"].idxmax() == modele_max
au_dela = tableau_tranches[(tableau_tranches["tranche"] == "> 125") & (tableau_tranches["modele"].isin(appris))].set_index("modele")
assert (au_dela["biais"] < 0).all() and int(au_dela["n"].iloc[0]) == 11
assert round(au_dela.loc["CNN 1D", "biais"], 2) == -8.51 and round(au_dela.loc["Random Forest", "biais"], 2) == -24.24

# classification a N = 30
n30 = tableau_classification[(tableau_classification["N"] == 30) & (tableau_classification["modele"].isin(appris))]
defaut = n30[n30["regle_seuil"] == "defaut (= N)"].set_index("modele")
choisi = n30[n30["regle_seuil"] == "choisi sur validation"].set_index("modele")
assert round(defaut["val_rappel"].min(), 2) == 0.90 and defaut["val_rappel"].idxmin() == "Random Forest"
assert round(defaut["val_rappel"].max(), 2) == 0.95 and defaut["val_rappel"].idxmax() == "LSTM"
assert round(defaut["val_precision"].min(), 2) == 0.96 and round(defaut["val_precision"].max(), 2) == 0.96
assert defaut["val_auc_roc"].min() >= 0.997
assert round(choisi["test_rappel"].min(), 2) == 0.92 and round(choisi["test_rappel"].max(), 2) == 1.00
naive_30 = tableau_classification[(tableau_classification["modele"] == "Reference naive") & (tableau_classification["N"] == 30)
                                  & (tableau_classification["regle_seuil"] == "defaut (= N)")].iloc[0]
assert round(naive_30["val_accuracy"], 3) == 0.850 and naive_30["val_rappel"] == 0
nb_positifs_15 = tableau_classification[tableau_classification["N"] == 15]["test_nb_positifs"].iloc[0]
assert nb_positifs_15 == 10

# comparaisons par paires
validation_pairs = comparaisons["validation"].set_index(["modele_a", "modele_b"])
assert validation_pairs.loc[("Random Forest", "LSTM"), "conclusion"] == "LSTM meilleur"
assert validation_pairs.loc[("LSTM", "CNN 1D"), "conclusion"] == "LSTM meilleur"
assert validation_pairs.loc[("Random Forest", "XGBoost"), "conclusion"] == "XGBoost meilleur"
assert validation_pairs.loc[("Random Forest", "LightGBM"), "conclusion"] == "LightGBM meilleur"
assert round(validation_pairs.loc[("XGBoost", "LSTM"), "p_wilcoxon_bonferroni"], 2) == 0.71
assert round(validation_pairs.loc[("LightGBM", "LSTM"), "p_wilcoxon_bonferroni"], 2) == 0.52
nb_paires_apprises = 0
for indice in comparaisons["test"].index:
    paire = comparaisons["test"].loc[indice]
    if paire["modele_a"] in appris and paire["modele_b"] in appris:
        nb_paires_apprises = nb_paires_apprises + 1
        assert paire["conclusion"] == "pas de difference significative"
assert nb_paires_apprises == 10
print("Les chiffres cites dans la section G sont bien ceux des calculs ci-dessus.")

<a id="partie-h"></a>
## H. Où nous situer dans la littérature

**Le problème** : beaucoup d'articles publient un RMSE sur le test C-MAPSS, mais tous ne mesurent pas exactement la même chose. Pour comparer honnêtement, il faut savoir (1) contre quelle vérité test l'erreur est calculée, (2) combien d'entraînements sont moyennés.

**Les articles retenus** (chiffres relus dans les articles eux-mêmes) :
- **Babu, Zhao, Li 2016** — *Deep Convolutional Neural Network Based Regression Approach for Estimation of Remaining Useful Life*, DASFAA 2016, LNCS 9642, p. 214-228, DOI [10.1007/978-3-319-32025-0_14](https://doi.org/10.1007/978-3-319-32025-0_14). Premier CNN sur C-MAPSS ; fenêtre de 15 cycles ; plafond de la RUL choisi par sous-jeu (valeur non donnée dans l'article).
- **Li, Ding, Sun 2018** — *Remaining useful life estimation in prognostics using deep convolution neural networks*, Reliability Engineering & System Safety 172 (2018), DOI [10.1016/j.ress.2017.11.021](https://doi.org/10.1016/j.ress.2017.11.021). DCNN ; fenêtre de 30 cycles et plafond de 125 sur FD001, moyenne de 10 entraînements. La version relue (prépublication de juin 2017, eScholarship) ne donne que FD001 ; DAST (Tableau III) et STAGNN (Tableau II) citent pour DCNN 22,36, 12,64 et 23,31 sur FD002 à FD004, tirés de la version publiée, non relue ici : ces trois chiffres ne sont pas repris dans le tableau.
- **Zhang, Song, Li 2022** — *Dual Aspect Self-Attention based on Transformer for Remaining Useful Life Prediction* (DAST), IEEE Transactions on Instrumentation and Measurement 71, DOI [10.1109/TIM.2022.3160561](https://doi.org/10.1109/TIM.2022.3160561). Fenêtre de 40 cycles sur FD001 et FD003, 60 sur FD002 et FD004, deux caractéristiques statistiques ajoutées (moyenne et coefficient de régression de la fenêtre), plafond 125, moyenne de 10 entraînements.
- **Huang, He, Sick 2023** — *Spatio-Temporal Attention Graph Neural Network for Remaining Useful Life Prediction* (STAGNN), CSCI 2023, arXiv:[2401.15964](https://arxiv.org/abs/2401.15964). Fenêtre de 50 cycles, plafond 125, moyenne de 10 essais ; la version STAGNN* normalise les capteurs **par régime de vol** (comme notre `src/sous_jeux.py` pour FD002 et FD004).

**La convention** : tous ces articles plafonnent les **étiquettes d'entraînement** (125 pour les trois derniers) ; **aucun ne dit si la vraie RUL du test est plafonnée** avant de calculer le RMSE. Nous donnons donc nos deux RMSE : contre la RUL brute (protocole officiel) et contre la RUL plafonnée à 125.

### H.1 Un indice sur la convention des articles : le plancher de la RUL brute

**L'idée** : un modèle entraîné sur une cible plafonnée à 125 ne prédit jamais beaucoup plus de 125. Même un modèle **parfait** sur la cible plafonnée (il prédit exactement min(vraie RUL, 125)) garde une erreur contre la RUL brute, à cause des moteurs test dont la vraie RUL dépasse 125. Ce RMSE minimal est un **plancher** : tant que les prédictions ne dépassent pas 125, aucun modèle ne descend en dessous. Un article qui publie un RMSE plus petit avec une cible plafonnée n'a donc très probablement pas mesuré contre la RUL brute (il faudrait que son modèle prédise nettement plus de 125 pour les moteurs concernés).

**Comment lire** : par sous-jeu, le nombre de moteurs test de vraie RUL > 125 et le plancher du RMSE contre la RUL brute.

In [ ]:
lignes = []
plancher_rmse_brute = {}  # sous-jeu -> RMSE contre la RUL brute du predicteur parfait plafonne a 125
for sous_jeu in SOUS_JEUX:
    rul_vraie_test = np.asarray(charger_rul_sous_jeu(sous_jeu), dtype=float).ravel()  # RUL_FD00X.txt
    prediction_parfaite = np.minimum(rul_vraie_test, PLAFOND_RUL)  # le meilleur modele possible sur la cible plafonnee
    plancher_rmse_brute[sous_jeu] = rmse(rul_vraie_test, prediction_parfaite)
    ligne = {}
    ligne["sous_jeu"] = sous_jeu
    ligne["moteurs_test"] = len(rul_vraie_test)
    ligne["moteurs_rul_vraie_sup_125"] = int(np.sum(rul_vraie_test > PLAFOND_RUL))
    ligne["vraie_rul_max"] = rul_vraie_test.max()
    ligne["plancher_rmse_brute"] = plancher_rmse_brute[sous_jeu]
    lignes.append(ligne)
tableau_plancher = pd.DataFrame(lignes).set_index("sous_jeu")
tableau_plancher.round(2)

### H.2 Le tableau de comparaison

**Ce qu'on fait** : les chiffres des articles (RMSE test, recopiés, jamais recalculés) à côté des nôtres, lus dans les résultats des sections F et G : pour chaque modèle retenu, RMSE contre la RUL plafonnée, puis contre la RUL brute. Pour les réseaux sur FD001, on ajoute la moyenne ± écart-type de nos 3 graines (42, 43, 44 ; écart-type à n − 1).

**Comment lire** : plus petit = mieux ; « - » = non publié pour ce sous-jeu, non calculé, ou absent de la version relue de l'article (DCNN sur FD002 à FD004, voir plus haut). Les articles moyennent 10 entraînements ; nous en avons un seul (la graine 42), ou 3 pour les réseaux sur FD001.

In [ ]:
# chiffres publies par les articles (RMSE test), recopies de leurs tableaux : ce ne sont pas des resultats du projet
ARTICLES = []
ARTICLES.append({"source": "Babu, Zhao, Li 2016", "modele": "CNN", "fenetre": "15", "entrainements": "non precise",
                 "FD001": 18.45, "FD002": 30.29, "FD003": 19.82, "FD004": 29.16})
ARTICLES.append({"source": "Li, Ding, Sun 2018", "modele": "DCNN", "fenetre": "30", "entrainements": "moyenne de 10",
                 "FD001": 12.61, "FD002": np.nan, "FD003": np.nan, "FD004": np.nan})
ARTICLES.append({"source": "Zhang, Song, Li 2022", "modele": "DAST", "fenetre": "40 / 60", "entrainements": "moyenne de 10",
                 "FD001": 11.43, "FD002": 15.25, "FD003": 11.32, "FD004": 18.36})
ARTICLES.append({"source": "Huang, He, Sick 2023", "modele": "STAGNN* (norm. par regime)", "fenetre": "50",
                 "entrainements": "moyenne de 10", "FD001": 11.50, "FD002": 13.81, "FD003": 11.05, "FD004": 14.30})

lignes = []
for article in ARTICLES:
    ligne = dict(article)  # copie : la liste ARTICLES reste intacte
    ligne["verite_test"] = "non precisee"
    lignes.append(ligne)

# nos modeles retenus (section F.7) : une ligne par convention de verite test
for nom_modele in ["LSTM", "CNN 1D", "XGBoost", "LightGBM", "Random Forest"]:
    for colonne, verite in [("test_plafonnee_retenu", "plafonnee a 125"), ("test_brute_retenu", "brute")]:
        ligne = {}
        ligne["source"] = "ce projet"
        ligne["modele"] = nom_modele
        if nom_modele in ["LSTM", "CNN 1D"]:
            ligne["fenetre"] = "30"
            ligne["entrainements"] = "1 (graine 42)"
        else:
            ligne["fenetre"] = "1 cycle (capteurs lisses sur 5, ecart aux 20 premiers)"
            ligne["entrainements"] = "1"
        for sous_jeu in SOUS_JEUX:
            ligne[sous_jeu] = round(tableau_test_sous_jeux.loc[(sous_jeu, nom_modele), colonne], 2)
        ligne["verite_test"] = verite
        lignes.append(ligne)

# reseaux sur FD001 : moyenne et ecart-type des 3 graines du modele final
tests_graines = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_test_fd001.csv"))
for cle in CLES_RESEAUX:
    graines = tests_graines[tests_graines["modele"] == cle]
    for colonne, verite in [("rmse_contre_rul_plafonnee", "plafonnee a 125"), ("rmse", "brute")]:
        ligne = {}
        ligne["source"] = "ce projet"
        ligne["modele"] = NOMS_RESEAUX[cle]
        ligne["fenetre"] = "30"
        ligne["entrainements"] = f"moyenne de {len(graines)} graines"
        ligne["FD001"] = formater_moyenne_ecart(graines[colonne].mean(), graines[colonne].std(), 2)  # std : n - 1
        for sous_jeu in ["FD002", "FD003", "FD004"]:
            ligne[sous_jeu] = np.nan
        ligne["verite_test"] = verite
        lignes.append(ligne)

tableau_litterature = pd.DataFrame(lignes)[["source", "modele", "fenetre", "entrainements", "verite_test",
                                             "FD001", "FD002", "FD003", "FD004"]]
tableau_litterature.fillna("-")  # "-" : non publie ou non calcule

### H.3 Ce qu'on peut dire, et ce qu'on ne peut pas dire

- **Le plancher tranche en partie la question de la convention.** Sur FD002 et FD004, un modèle parfait sur la cible plafonnée garderait un RMSE de 18,32 et 19,53 contre la RUL brute. DAST (15,25 et 18,36) et STAGNN* (13,81 et 14,30) publient moins que ce plancher : avec des étiquettes plafonnées à 125, ils n'ont pu mesurer contre la RUL brute que si leurs modèles prédisent nettement plus de 125 pour les moteurs concernés. Ils comparent donc très probablement à la vraie RUL **plafonnée** : c'est notre RMSE plafonné qu'il faut mettre en face. C'est une **hypothèse** (les articles ne l'écrivent pas) ; sur FD001 et FD003, le plancher (3,74 et 4,71) ne permet pas de trancher.
- **FD001, sous cette hypothèse** : notre LSTM (11,37 avec la graine 42 ; 11,26 ± 0,43 sur 3 graines) et notre XGBoost (11,88) sont **au niveau** de DAST (11,43), STAGNN* (11,50) et DCNN (12,61), et nettement meilleurs que le premier CNN publié (Babu, 18,45, dont le plafond n'est pas donné). L'écart entre notre LSTM et DAST (0,06 cycle) est plus petit que la variation de notre LSTM d'une graine à l'autre (0,43) : on ne peut **pas** dire que nous faisons mieux.
- **FD002 et FD004** : notre RMSE plafonné (LSTM 11,05 et 10,95 ; XGBoost 11,47 et 12,57) est plus bas que celui de STAGNN* (13,81 et 14,30). À confirmer avant de l'affirmer : la convention reste une hypothèse, et c'est un seul entraînement contre une moyenne de 10.
- **Mise en garde générale** : les articles moyennent 10 entraînements ; nous en avons un seul par modèle (la graine 42), ou 3 pour les réseaux sur FD001. Un bon chiffre isolé peut venir d'une graine favorable : il faudrait 10 graines pour une comparaison à armes égales.

**Contrôle des chiffres de la section H** : la cellule suivante vérifie les chiffres cités ci-dessus.

In [ ]:
assert round(plancher_rmse_brute["FD001"], 2) == 3.74 and round(plancher_rmse_brute["FD003"], 2) == 4.71
assert round(plancher_rmse_brute["FD002"], 2) == 18.32 and round(plancher_rmse_brute["FD004"], 2) == 19.53
articles = pd.DataFrame(ARTICLES).set_index("modele")
for sous_jeu in ["FD002", "FD004"]:  # DAST et STAGNN* publient moins que le plancher de la RUL brute
    assert articles.loc["DAST", sous_jeu] < plancher_rmse_brute[sous_jeu]
    assert articles.loc["STAGNN* (norm. par regime)", sous_jeu] < plancher_rmse_brute[sous_jeu]

graines_lstm = tests_graines[tests_graines["modele"] == "lstm"]["rmse_contre_rul_plafonnee"]
assert round(graines_lstm.mean(), 2) == 11.26 and round(graines_lstm.std(), 2) == 0.43
assert round(tableau_test_sous_jeux.loc[("FD001", "LSTM"), "test_plafonnee_retenu"], 2) == 11.37
assert round(tableau_test_sous_jeux.loc[("FD001", "XGBoost"), "test_plafonnee_retenu"], 2) == 11.88
assert round(tableau_test_sous_jeux.loc[("FD001", "LSTM"), "test_plafonnee_retenu"] - articles.loc["DAST", "FD001"], 2) == -0.06
for sous_jeu, lstm_attendu, xgb_attendu in [("FD002", 11.05, 11.47), ("FD004", 10.95, 12.57)]:
    assert round(tableau_test_sous_jeux.loc[(sous_jeu, "LSTM"), "test_plafonnee_retenu"], 2) == lstm_attendu
    assert round(tableau_test_sous_jeux.loc[(sous_jeu, "XGBoost"), "test_plafonnee_retenu"], 2) == xgb_attendu
    assert lstm_attendu < articles.loc["STAGNN* (norm. par regime)", sous_jeu]
    assert xgb_attendu < articles.loc["STAGNN* (norm. par regime)", sous_jeu]
print("Les chiffres cites dans la section H sont bien ceux des tableaux ci-dessus.")

<a id="partie-i"></a>
## I. Synthèse

### I.1 Tableau final des modèles à réglages de base (sections B et D)

**Comment lire** : pour chaque modèle à arbres de la partie 1 (46 variables), les colonnes de validation croisée (mêmes 5 plis) puis celles du test officiel (évalué une fois). Les versions retenues sont résumées en F.7 (tous sous-jeux) et en G.3 (FD001, toutes les métriques).

In [ ]:
lignes = []
for nom_modele in resultats_test:  # meme ordre que le tableau de la section D
    ligne_cv = resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN)  # validation croisee (section B)
    ligne_test = resumer_test(nom_modele, resultats_test[nom_modele])  # test officiel (section D)
    ligne = {}
    ligne["modele"] = nom_modele
    ligne["rmse_cv"] = ligne_cv["rmse_cv_moyen"]
    ligne["nasa_cv"] = ligne_cv["nasa_cv_moyen"]
    ligne["surestimes_oof"] = ligne_cv["surestimes_oof"]
    ligne["rmse_test"] = ligne_test["rmse_test"]
    ligne["rmse_test_plafonnee"] = ligne_test["rmse_test_rul_plafonnee"]
    ligne["nasa_test"] = ligne_test["nasa_test"]
    ligne["surestimes_test"] = ligne_test["surestimes_test"]
    ligne["alertes_manquees_test"] = ligne_test["alertes_manquees_test"]
    ligne["fausses_alertes_test"] = ligne_test["fausses_alertes_test"]
    lignes.append(ligne)
tableau_synthese = pd.DataFrame(lignes).set_index("modele")
tableau_synthese.round(2)

### I.2 Tableaux de résultats

**Ce qu'on fait** : on rassemble les résultats de la partie 1 dans deux tableaux : `tableau_comparaison_cv` (les modèles des sections B et C, une ligne par modèle) et `tableau_comparaison_test` (les trois modèles évalués sur le test, section D). Ils servent au contrôle de la section I.5.

In [ ]:
# 1) validation croisee : sections B et C, une ligne par modele
lignes = []
for nom_modele in resultats_cv_base:
    ligne = resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN)
    ligne["section"] = "B - reglages de base"
    lignes.append(ligne)
for reglage in reglages_penalite:
    ligne = resumer_cv(reglage["reglage"], resultats_cv_penalite[reglage["reglage"]], NB_MOTEURS_TRAIN)
    ligne["section"] = "C - penalite de surestimation"
    lignes.append(ligne)
tableau_comparaison_cv = pd.DataFrame(lignes)

# 2) test officiel : section D
tableau_comparaison_test = tableau_test.reset_index()  # "modele" redevient une colonne ordinaire
print(f"{len(tableau_comparaison_cv)} modeles en validation croisee, {len(tableau_comparaison_test)} sur le test")

### I.3 Ce qui est établi, ce qui ne l'est pas

**Établi** (validation croisée, mêmes 5 plis, tests appariés sur 100 moteurs, règles écrites avant les résultats)
- À réglages de base, les trois ensembles d'arbres battent largement la référence naïve et sont à égalité entre eux (section B).
- Changer les constantes 13/10 du score ne change aucune prédiction ; pénaliser la surestimation **dans l'apprentissage** la réduit, avec un coût en RMSE, en fausses alertes et en biais (section C).
- L'écart à l'état initial `ecartinit_20` (Matisse) améliore nettement les 5 modèles qui apprennent, sans dégrader la prudence ; la fenêtre de 50 cycles n'apporte rien de plus aux réseaux (section F).
- Les 5 modèles qui apprennent battent les 5 baselines du protocole en RMSE, MAE, R² et score NASA, en validation comme au test, et toutes leurs comparaisons appariées contre Ridge et la référence naïve sont significatives ; en validation, le LSTM est significativement meilleur que Random Forest et que le CNN 1D (section G).
- Les chiffres de test de la Phase 3 sont retrouvés à l'identique (section D), ceux de sa validation croisée à l'arrondi près (16,87 contre 16,86 publié, note de la section B) ; ceux des scripts à l'identique (G.1).

**Non concluant**
- Sur le test (100 moteurs), aucune différence significative entre modèles appris : ni entre les trois arbres à réglages de base (section D), ni entre les 5 versions retenues (section G.6).
- La position par rapport aux articles dépend d'une convention qu'ils n'écrivent pas (section H) ; nos chiffres viennent d'un seul entraînement (3 graines pour les réseaux sur FD001), contre 10 dans les articles.
- Les hypothèses H1 et H2 sur les moteurs surestimés (section E) : trop peu de moteurs pour conclure.

### I.4 Limites et suite
- Choix des améliorations faits sur FD001 seulement, appliqués tels quels à FD002–FD004 ; sur FD004, le CNN 1D retenu se dégrade fortement au test (section F.8) : constat fait sur le test, qui ne change pas le choix (règle 1) ; à examiner en validation, et à signaler quand la démo montre ce modèle.
- Hyperparamètres non réoptimisés pour le jeu de 31 variables ; un seul entraînement par modèle (3 graines pour les réseaux FD001), contre 10 dans les articles.
- Étapes du protocole non faites, listées en G.7 (erreur d'entraînement des 6 modèles, classifieur dédié, courbes d'apprentissage, hyperparamètres, validation croisée répétée, protocole sur FD002–FD004).
- Seuils d'alerte (30 cycles) et de surestimation (20 cycles) : hypothèses de travail, à discuter. La clé `xgboost_phase3` des fichiers et de la démo désigne maintenant la version à 31 variables (nom gardé pour le contrat des fichiers).
- La validation croisée note tous les cycles, le test seulement le dernier cycle d'une trajectoire tronquée : une validation qui imite cette troncature serait plus fidèle.

### I.5 Contrôle des chiffres de l'encadré « En 30 secondes »

**Ce qu'on fait** : on vérifie que les chiffres cités dans l'encadré du début sont bien ceux des tableaux calculés ou relus dans ce notebook. **Pourquoi** : si un résultat change un jour, cette cellule s'arrête sur une erreur au lieu de laisser un texte faux en tête du notebook.

In [ ]:
# partie 1 : les deux tableaux de la section I.2 ; la colonne "modele" sert d'etiquette de ligne
cv_relu = tableau_comparaison_cv.set_index("modele")
test_relu = tableau_comparaison_test.set_index("modele")

# section C : XGBoost, perte quadratique asymetrique, alpha = 1 puis alpha = 3
alpha_1 = cv_relu.loc["XGBoost quadratique alpha=1"]
alpha_3 = cv_relu.loc["XGBoost quadratique alpha=3"]
baisse_surestimes = 100.0 * (1.0 - alpha_3["surestimes_oof"] / alpha_1["surestimes_oof"])  # en pourcentage
print(f"Section C, alpha 1 -> 3 : surestimes OOF > 20 -{baisse_surestimes:.0f} %, "
      f"RMSE CV {alpha_1['rmse_cv_moyen']:.2f} -> {alpha_3['rmse_cv_moyen']:.2f}")
assert round(baisse_surestimes) == 33
assert round(alpha_1["rmse_cv_moyen"], 2) == 16.86 and round(alpha_3["rmse_cv_moyen"], 2) == 17.68

# section D : test officiel de XGBoost Phase 3 (version de base), calcule en direct
test_phase3 = test_relu.loc["XGBoost Phase 3"]
print(f"Test FD001, XGBoost Phase 3 (base) : RMSE brute {test_phase3['rmse_test']:.2f}, "
      f"plafonnee {test_phase3['rmse_test_rul_plafonnee']:.2f}")
assert round(test_phase3["rmse_test"], 2) == 19.70 and round(test_phase3["rmse_test_rul_plafonnee"], 2) == 18.53

# section F : ameliorations
gain_lstm = candidats_indexes.loc[("lstm", "A"), "gain_rmse"]
gain_xgb = choix_fd001.loc["xgboost_phase3", "baisse_rmse_cv"]
print(f"Section F : baisse du RMSE CV {gain_lstm:.2f} (LSTM) a {gain_xgb:.2f} (XGBoost)")
assert round(gain_lstm, 2) == 4.76 and round(gain_xgb, 2) == 7.51
assert round(min(baisses), 2) == 4.76 and round(max(baisses), 2) == 7.51  # extremes des 5 modeles (controle F)
assert round(candidats_indexes.loc[("lstm", "base"), "rmse_cv_moyen"], 2) == 13.70
assert round(candidats_indexes.loc[("lstm", "A"), "rmse_cv_moyen"], 2) == 8.94
assert round(choix_fd001.loc["xgboost_phase3", "rmse_cv_base"], 2) == 16.87
assert round(choix_fd001.loc["xgboost_phase3", "rmse_cv_candidat"], 2) == 9.36
assert decisions_reseaux["decisions"]["lstm"]["longueur_fenetre"] == 30
assert decisions_reseaux["decisions"]["cnn1d"]["longueur_fenetre"] == 30

# test officiel FD001 des versions retenues
for nom_modele, brute, plafonnee in [("LSTM", 12.29, 11.37), ("XGBoost", 13.10, 11.88)]:
    assert round(tableau_test_sous_jeux.loc[("FD001", nom_modele), "test_brute_retenu"], 2) == brute
    assert round(tableau_test_sous_jeux.loc[("FD001", nom_modele), "test_plafonnee_retenu"], 2) == plafonnee

# section G : baselines et comparaisons sur le test
assert round(resume_validation["Ridge"]["rmse_moyenne"], 2) == 16.95
assert nb_paires_apprises == 10  # 10 paires entre modeles appris, aucune significative (controle G)

# section H : litterature
assert articles.loc["DAST", "FD001"] == 11.43 and articles.loc["STAGNN* (norm. par regime)", "FD001"] == 11.50
assert articles.loc["DAST", "FD002"] < plancher_rmse_brute["FD002"]
assert round(graines_lstm.mean(), 2) == 11.26 and round(graines_lstm.std(), 2) == 0.43  # 3 graines (H.2)

# limite : CNN 1D sur FD004
assert round(cnn_fd004["nasa_test_retenu"]) == 446374 and cnn_fd004["surestimes_test"] == 55 and cnn_fd004["moteurs_test"] == 248
print("Les chiffres de l'encadre sont bien ceux des tableaux de resultats.")

**Durée** : temps de calcul des sections A à I (la section J, interactive, est chronométrée à part).

In [ ]:
duree_notebook = time.perf_counter() - debut_notebook  # secondes ecoulees depuis la cellule A.2
print(f"Sections A a I executees en {duree_notebook / 60:.1f} minutes.")

<a id="partie-j"></a>
## J. Démo : tester un moteur que les modèles n'ont jamais vu

**Le principe** : les modèles ont appris uniquement sur `train_FD00X.txt`. On tire au hasard un moteur de `test_FD00X.txt` ; à chaque vol (un cycle), le modèle ne reçoit que les mesures jusqu'à ce vol, comme en service. La vraie RUL, lue dans le fichier `RUL_FD00X.txt` de la NASA, n'est montrée qu'après.
**Comment** : `src/inference.py` fait tout le calcul en direct (préparation apprise sur le train, puis prédiction) ; aucun fichier de prédictions précalculées n'est lu.
**Astuce** : cette section peut s'exécuter seule, après un redémarrage du noyau : sa première cellule refait les imports nécessaires.

### J.1 Réglages de la démo

**À modifier avant de lancer** : `SOUS_JEU` (FD001 à FD004) ; `GRAINE` (`None` = un moteur différent à chaque exécution de J.3, un nombre = toujours le même moteur, pour rejouer une démo) ; la liste des modèles comparés : XGBoost (version à 31 variables), LSTM et CNN 1D.

In [ ]:
import os
import random
import sys
import time

# racine du depot importable (utile si cette section est lancee seule, apres un redemarrage du noyau)
RACINE_DEPOT_DEMO = os.path.abspath("..")  # le notebook est dans notebooks/ : la racine est le dossier parent
if RACINE_DEPOT_DEMO not in sys.path:  # on ne l'ajoute qu'une fois
    sys.path.insert(0, RACINE_DEPOT_DEMO)

import matplotlib.pyplot as plt

# ORDRE VOULU : src.inference importe xgboost AVANT torch, puis regle torch sur 1 coeur (voir src/deep_learning.py) ;
# importer torch soi-meme avant xgboost peut bloquer ou faire planter le noyau
from src import inference

SOUS_JEU = "FD001"  # sous-jeu NASA : "FD001", "FD002", "FD003" ou "FD004"
GRAINE = None       # None : un moteur different a chaque execution de J.3 ; un entier (ex. 7) : toujours le meme moteur
MODELES_DEMO = ["xgboost_phase3", "lstm", "cnn1d"]  # cles des modeles dans src/inference.py
SEUIL_ALERTE_DEMO = 30  # cycles : RUL predite <= 30 -> alerte maintenance (hypothese de travail, comme en A.2)
COULEURS_DEMO = ["#2a78d6", "#eb6834", "#1baf7a"]  # bleu, orange, vert d'eau (memes teintes qu'en A.2)
debut_demo = time.perf_counter()  # chronometre de la section J

### J.2 Chargement des modèles

**Ce qu'on fait** : `charger_modeles` apprend la préparation sur `train_<SOUS_JEU>` (capteurs retirés, min / max des capteurs, régimes de vol), relit XGBoost (version à 31 variables) dans `models/` (ou le réentraîne en mémoire s'il manque ou ne correspond plus à la préparation, sans rien écrire sur le disque) et relit les poids du LSTM et du CNN 1D (jamais réentraînés ici). **Comment lire** : un modèle indisponible (fichier absent) est signalé avec la raison, les autres fonctionnent.

In [ ]:
# sauvegarder=False : si XGBoost doit etre reentraine, il reste en memoire (le notebook n'ecrit rien dans models/)
inference_demo = inference.charger_modeles(SOUS_JEU, modeles=MODELES_DEMO, sauvegarder=False)

print(f"Sous-jeu {SOUS_JEU} : chargement en {inference_demo['duree_chargement_s']:.1f} s")
for modele in inference_demo["modeles"]:  # les modeles prets a predire
    print(f"  disponible   : {inference.nom_modele(modele):16s} ({inference_demo['modeles'][modele]['description']})")
for modele in inference_demo["indisponibles"]:  # par exemple un fichier de poids absent
    print(f"  indisponible : {inference.nom_modele(modele):16s} -> {inference_demo['indisponibles'][modele]}")
for message in inference_demo["messages"]:  # par exemple : XGBoost reentraine en memoire faute de fichier
    print("  note :", message)

### J.3 Prédiction en direct sur un moteur mystère

**Ce qu'on fait** : on tire un moteur de test au hasard sans montrer son numéro. (1) Au dernier vol observé n, chaque modèle reçoit les mesures des vols 1 à n et prédit la RUL, avec son temps de calcul. (2) On refait la prédiction à chaque vol t, avec les seules mesures 1 à t, pour tracer une courbe.
**Comment lire** : une courbe par modèle ; sous la ligne pointillée (30 cycles), le modèle déclencherait une alerte maintenance. Relancer cette cellule tire un nouveau moteur (si `GRAINE = None`).

In [ ]:
def tracer_courbes_demo(trajectoire, modeles, titre, verite=None):
    """
    Trace la RUL predite a chaque vol par chaque modele, et la vraie RUL si elle est fournie.

    Parameters
    ----------
    trajectoire : pandas.DataFrame
        Le resultat de `inference.predire_trajectoire` : "time_cycles" et une colonne par modele.
    modeles : list de str
        Les cles des modeles a tracer (colonnes de `trajectoire`).
    titre : str
        Le titre de la figure.
    verite : pandas.DataFrame ou None, optionnel
        Le resultat de `inference.verite_moteur` ("time_cycles", "rul_vraie", "rul_vraie_cappee") ;
        None (defaut) : la verite n'est pas dessinee.

    Returns
    -------
    None
    """
    fig, ax = plt.subplots(figsize=(13, 5))
    for numero, modele in enumerate(modeles):  # une courbe et une couleur par modele
        couleur = COULEURS_DEMO[numero % len(COULEURS_DEMO)]  # % : on reprend la liste au debut s'il manque des couleurs
        ax.plot(trajectoire["time_cycles"], trajectoire[modele], color=couleur, linewidth=2,
                label=f"RUL prédite — {inference.nom_modele(modele)}")
    if verite is not None:  # la verite n'est dessinee qu'une fois revelee (cellule J.4)
        ax.plot(verite["time_cycles"], verite["rul_vraie"], color="black", linewidth=2,
                label=f"vraie RUL (RUL_{SOUS_JEU}.txt)")
        ax.plot(verite["time_cycles"], verite["rul_vraie_cappee"], color="black", linestyle="--", linewidth=1,
                label="vraie RUL plafonnée à 125 (la cible apprise)")
    ax.axhline(SEUIL_ALERTE_DEMO, color="#52514e", linestyle=":", label=f"seuil d'alerte ({SEUIL_ALERTE_DEMO} cycles)")
    ax.set_ylim(bottom=0)  # l'axe vertical part de 0 cycle restant
    ax.set_xlabel("vol (cycle) observé")
    ax.set_ylabel("RUL (cycles restants)")
    ax.set_title(titre)
    ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1.0))  # legende a droite, hors des courbes
    plt.tight_layout()
    plt.show()


# 1) tirage du moteur mystere
if GRAINE is None:
    # alea fourni par le systeme d'exploitation : jamais influence par une graine fixee ailleurs dans le notebook
    graine_tirage = random.SystemRandom().randrange(1_000_000)  # entier entre 0 et 999 999
else:
    graine_tirage = GRAINE
moteur_mystere = inference.tirer_moteur_test(SOUS_JEU, graine=graine_tirage)  # meme graine = meme moteur
mesures_mystere = moteur_mystere["mesures"]  # mesures brutes des vols 1 a n (colonnes de test_<SOUS_JEU>.txt)
nb_vols = moteur_mystere["nb_cycles"]  # n : nombre de vols observes

print(f"Moteur mystere de test_{SOUS_JEU}.txt : numero masque, {nb_vols} vols observes")
print(f"Graine du tirage : {graine_tirage} (mettre GRAINE = {graine_tirage} en J.1 pour rejouer ce moteur)")

# 2) prediction au dernier vol observe : chaque modele recoit les mesures des vols 1 a n, rien d'autre
au_dernier_vol = inference.predire_au_cycle(inference_demo, mesures_mystere, nb_vols)
print(f"\nRUL predite au vol {nb_vols} (preparation des mesures : {au_dernier_vol['temps_preparation_ms']:.1f} ms) :")
for modele in au_dernier_vol["predictions"]:  # un modele a la fois
    print(f"  {inference.nom_modele(modele):16s} : {au_dernier_vol['predictions'][modele]:6.1f} cycles "
          f"(calcul : {au_dernier_vol['temps_ms'][modele]:.1f} ms)")

# 3) prediction a chaque vol t, avec les seules mesures 1 a t (une troncature et une preparation par vol)
debut = time.perf_counter()
trajectoire_mystere = inference.predire_trajectoire(inference_demo, mesures_mystere)  # une ligne par vol
duree_trajectoire = time.perf_counter() - debut
modeles_traces = list(au_dernier_vol["predictions"].keys())  # les modeles disponibles, dans l'ordre d'affichage
print(f"\nCourbe : {nb_vols} vols x {len(modeles_traces)} modeles recalcules en {duree_trajectoire:.1f} s "
      f"(environ {1000 * duree_trajectoire / nb_vols:.0f} ms par vol)")

tracer_courbes_demo(trajectoire_mystere, modeles_traces, f"Moteur mystère ({SOUS_JEU}) : RUL prédite à chaque vol")

### J.4 Révéler la vérité

**Ce qu'on fait** : on affiche le numéro du moteur, sa vraie RUL au dernier vol (`RUL_<SOUS_JEU>.txt`), l'écart de chaque modèle (positif = surestimation, le cas dangereux) et son statut d'alerte, puis la courbe avec la vérité.
**Règle d'alerte** (la même que `classer_alerte` de `src/alertes.py` et que les colonnes d'alertes des sections B à D) : le modèle alerte si sa RUL prédite ≤ 30 ; il aurait dû alerter si la vraie RUL ≤ 30. Les modèles ont appris une cible plafonnée à 125 : si la vraie RUL dépasse 125, une prédiction plus basse (écart négatif) est attendue.

In [ ]:
def statut_alerte(rul_predite, rul_vraie, seuil):
    """
    Classe une prediction en quatre cas, avec la meme regle que `classer_alerte` de src/alertes.py.

    Le modele declenche une alerte si la RUL predite est <= seuil ; il aurait du alerter si la vraie
    RUL est <= seuil.

    Parameters
    ----------
    rul_predite : float
        La RUL predite (cycles).
    rul_vraie : float
        La vraie RUL (cycles).
    seuil : float
        Le seuil d'alerte (cycles).

    Returns
    -------
    str
        "alerte correcte", "ALERTE MANQUEE" (le cas dangereux), "fausse alerte" ou "rien a signaler".
    """
    alerte_declenchee = rul_predite <= seuil  # decision du modele
    aurait_du_alerter = rul_vraie <= seuil  # ce qu'il aurait fallu faire
    if alerte_declenchee and aurait_du_alerter:
        return "alerte correcte"
    if aurait_du_alerter:  # il fallait alerter et le modele ne l'a pas fait
        return "ALERTE MANQUEE"
    if alerte_declenchee:  # alerte alors que le moteur avait encore de la marge (maintenance inutile)
        return "fausse alerte"
    return "rien a signaler"


rul_vraie_finale = moteur_mystere["rul_finale"]  # ligne de ce moteur dans RUL_<SOUS_JEU>.txt
print(f"Le moteur mystere etait le moteur n°{moteur_mystere['moteur']} de test_{SOUS_JEU}.txt")
print(f"Vraie RUL au vol {nb_vols} : {rul_vraie_finale:.0f} cycles (RUL_{SOUS_JEU}.txt)\n")
for modele in au_dernier_vol["predictions"]:
    rul_predite = au_dernier_vol["predictions"][modele]
    ecart = rul_predite - rul_vraie_finale  # positif = surestimation
    statut = statut_alerte(rul_predite, rul_vraie_finale, SEUIL_ALERTE_DEMO)
    print(f"  {inference.nom_modele(modele):16s} : predit {rul_predite:6.1f} cycles, ecart {ecart:+6.1f} -> {statut}")

verite_mystere = inference.verite_moteur(mesures_mystere, rul_vraie_finale)  # vraie RUL a chaque vol
tracer_courbes_demo(trajectoire_mystere, modeles_traces,
                    f"Moteur n°{moteur_mystere['moteur']} ({SOUS_JEU}) : prédictions et vérité", verite=verite_mystere)

### J.5 Lancer la démo interactive (application Streamlit)

**Ce qu'on fait** : on démarre `app/tableau_de_bord.py` en arrière-plan avec le Python du projet (`.venv`), depuis la racine du dépôt, sur le premier port libre à partir de 8599 : le notebook reste utilisable pendant que l'application tourne.
**Comment** : `subprocess.Popen` lance l'application sans attendre sa fin ; on interroge son adresse `/_stcore/health` jusqu'à la réponse « ok », puis on affiche le lien à ouvrir dans le navigateur.
**Attention** : « Exécuter tout » démarre puis arrête aussitôt l'application (dernière cellule). Pour la présentation, relancer cette cellule seule.

In [ ]:
import atexit          # pour arreter l'application si Python se ferme sans passer par la cellule J.6
import socket          # pour tester si un port est libre
import subprocess      # pour lancer un autre programme (l'application) sans bloquer le notebook
import tempfile        # dossier temporaire du systeme, pour le journal de l'application
import urllib.request  # pour interroger l'application par HTTP

from IPython.display import Markdown, display  # pour afficher un lien cliquable

PORT_DE_DEPART = 8599  # premier port essaye ; s'il est deja pris (par une autre application), on essaie le suivant
DELAI_DEMARRAGE_MAX_S = 60  # temps maximal d'attente du demarrage, en secondes


def port_libre(port):
    """
    Dit si un port de cet ordinateur est libre, en essayant de le reserver un court instant.

    Parameters
    ----------
    port : int
        Le numero de port.

    Returns
    -------
    bool
        Vrai si aucun programme n'utilise ce port.
    """
    essai = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        essai.bind(("127.0.0.1", port))  # echoue si un autre programme utilise deja ce port
        libre = True
    except OSError:
        libre = False
    essai.close()  # on libere le port tout de suite
    return libre


def premier_port_libre(port_depart, nb_essais=50):
    """
    Renvoie le premier port libre a partir de port_depart.

    Parameters
    ----------
    port_depart : int
        Le premier port essaye.
    nb_essais : int, optionnel
        Nombre de ports essayes au maximum (50 par defaut).

    Returns
    -------
    int
        Le premier port libre.
    """
    for port in range(port_depart, port_depart + nb_essais):  # 8599, 8600, 8601, ...
        if port_libre(port):
            return port
    raise RuntimeError(f"Aucun port libre entre {port_depart} et {port_depart + nb_essais - 1}.")


def arreter_processus(processus):
    """
    Arrete l'application : demande d'arret, puis arret force si elle ne s'est pas arretee en 10 secondes.

    Parameters
    ----------
    processus : subprocess.Popen
        Le processus lance par `subprocess.Popen`.

    Returns
    -------
    str
        "deja arretee", "arretee" ou "arretee de force".
    """
    if processus.poll() is not None:  # poll() renvoie None tant que le processus tourne
        return "deja arretee"
    processus.terminate()  # demande d'arret (signal SIGTERM)
    try:
        processus.wait(timeout=10)  # on attend la fin, au plus 10 secondes
        return "arretee"
    except subprocess.TimeoutExpired:
        processus.kill()  # arret force (signal SIGKILL)
        processus.wait()
        return "arretee de force"


def attendre_application(processus, url_sante, delai_max_s):
    """
    Interroge l'adresse de sante de l'application jusqu'a la reponse "ok".

    Parameters
    ----------
    processus : subprocess.Popen
        Le processus de l'application (pour detecter un arret premature).
    url_sante : str
        L'adresse a interroger (".../_stcore/health").
    delai_max_s : float
        Temps maximal d'attente, en secondes.

    Returns
    -------
    bool
        Vrai si l'application a repondu "ok" a temps.
    """
    debut = time.perf_counter()
    while time.perf_counter() - debut < delai_max_s:
        if processus.poll() is not None:  # l'application s'est deja arretee : erreur au demarrage
            return False
        try:
            with urllib.request.urlopen(url_sante, timeout=2) as reponse:
                if reponse.read().decode().strip() == "ok":
                    return True
        except OSError:  # pas encore prete (connexion refusee) : on reessaie un peu plus tard
            pass
        time.sleep(0.5)  # demi-seconde entre deux essais
    return False


# python du projet : .venv/bin/python s'il existe (macOS, Linux), sinon le python de ce noyau
python_demo = os.path.join(RACINE_DEPOT_DEMO, ".venv", "bin", "python")
if not os.path.exists(python_demo):
    python_demo = sys.executable

port_demo = premier_port_libre(PORT_DE_DEPART)
url_demo = f"http://127.0.0.1:{port_demo}"
chemin_journal = os.path.join(tempfile.gettempdir(), f"aero_predict_streamlit_{port_demo}.log")  # messages de l'application
journal = open(chemin_journal, "w", encoding="utf-8")

commande = [python_demo, "-m", "streamlit", "run", "app/tableau_de_bord.py",
            "--server.port", str(port_demo),
            "--server.address", "127.0.0.1",        # accessible depuis cet ordinateur seulement
            "--server.headless", "true",            # n'ouvre pas de navigateur tout seul
            "--browser.gatherUsageStats", "false"]  # aucune statistique d'usage envoyee
# Popen lance la commande et rend la main tout de suite ; cwd : depuis la racine du depot
processus_demo = subprocess.Popen(commande, cwd=RACINE_DEPOT_DEMO, stdout=journal, stderr=subprocess.STDOUT)
atexit.register(arreter_processus, processus_demo)  # filet de securite : arret quand le noyau Python se ferme

if attendre_application(processus_demo, f"{url_demo}/_stcore/health", DELAI_DEMARRAGE_MAX_S):
    print(f"Application demarree (processus {processus_demo.pid}, port {port_demo}, journal : {chemin_journal}).")
    display(Markdown(f"**Démo prête** : [{url_demo}]({url_demo}) — à ouvrir dans le navigateur."))
else:
    arreter_processus(processus_demo)
    journal.close()
    with open(chemin_journal, encoding="utf-8") as fichier:
        print(fichier.read()[-2000:])  # les 2 000 derniers caracteres du journal, pour comprendre l'erreur
    raise RuntimeError(f"L'application n'a pas repondu en {DELAI_DEMARRAGE_MAX_S} s (journal ci-dessus).")

### J.6 Arrêter la démo

**Ce qu'on fait** : on arrête l'application lancée en J.5 (demande d'arrêt, puis arrêt forcé si elle ne répond pas en 10 secondes) et on ferme son journal. À lancer à la fin de la présentation ; c'est aussi la dernière cellule du notebook, pour qu'une exécution complète ne laisse aucun programme tourner.

In [ ]:
etat = arreter_processus(processus_demo)  # sans effet si l'application est deja arretee
journal.close()  # ferme le fichier des messages de l'application
print(f"Application du port {port_demo} : {etat}.")
print(f"Section J executee en {time.perf_counter() - debut_demo:.0f} s.")